| | |
|---|---|
| **CURSO** | Herramientas de Big Data |
| **SABER 2** | Procesamiento distribuido |
| **NOTEBOOK** | Procesamiento distribuido: benchmark y PoC en PySpark |
| **ACTIVIDAD** | Actividad 1 (análisis comparativo) y Actividad 2 (arquitectura y PoC en PySpark) |
| **DATASET** | Llamadas de urgencias y emergencias de la Línea 123 de Bogotá (Secretaría Distrital de Salud), julio–septiembre de 2026: 3 CSV, 27 434 registros, ~3 MB |
| **TIEMPO ESTIMADO** | ~30 minutos en Colab (runtime CPU) |

**Objetivos de aprendizaje:**
- Medir de forma automatizada tiempo y memoria de Pandas, Polars y PySpark con measure_performance().
- Construir un pipeline PySpark con ingesta desde Drive, validación de calidad y exportación a Parquet.
- Medir el PoC tres veces y compararlo con una implementación equivalente en Pandas.
- Analizar planes de ejecución, particionamiento, formatos y escalabilidad.

**Prerrequisitos:**
- Cuenta de Google con Drive; runtime CPU gratuito de Colab.
- Conocimientos básicos de DataFrames en Pandas y PySpark.

# Procesamiento distribuido de las emergencias de la Línea 123 de Bogotá

Este notebook reúne **todo lo que se hizo** en la Unidad 2, en el orden en que se hizo, con el código y sus
resultados. Es el único programa de análisis del proyecto; el segundo notebook (`U2_informes_y_entregables.ipynb`)
solo convierte las mediciones que deja este en informes.

**Dónde está cada tarea de la guía.**

| Tarea de la guía | Dónde |
|---|---|
| **1.1** Benchmark Pandas / Polars / PySpark con `measure_performance()` | Sección 3 (3.0 a 3.5) |
| **1.2** Matriz de decisión ponderada | Sección 3.7 y notebook de informes (Excel con fórmulas) |
| **1.3** Selección tecnológica justificada | Sección 3.7 y `analisis_decision_tecnologica.docx` |
| **2.1** Arquitectura y diagrama | Notebook de informes (`arquitectura_tecnica.docx`, `.png`, `.drawio`) |
| **2.2** PoC en PySpark: ingesta desde Drive, validaciones, Parquet y log | Sección 4 |
| **2.4** Métricas de 3 ejecuciones frente a Pandas y escalabilidad | Secciones 4.3, 5 y notebook de informes |

**Recorrido.** Entorno y configuración (0–1) · datos (2) · **Actividad 1**: benchmark (3) · **Actividad 2**: PoC
(4), particionamiento y escalabilidad (5), optimización con los patrones del curso (6) · pruebas (7) · conclusiones (8).

**Cómo leer este notebook.** Las secciones 2 a 8 cuentan el trabajo en el orden de la guía y **muestran en pantalla
el código que importa**: `measure_performance()`, las 5 operaciones de cada herramienta y el pipeline de PySpark. El
**Anexo A** (sección plegada, antes de la 1) escribe en el disco del VM el código de apoyo; hay que ejecutarlo primero
(*Ejecutar todas* lo hace). Está en archivos porque `measure_performance()` ejecuta cada repetición en un
**proceso Python nuevo**, para que ninguna herramienta herede cachés ni una JVM ya caliente, y ese proceso importa el
código desde un archivo.

**Regla de limpieza.** Solo se eliminan las **copias duplicadas exactas** (100 % de sus atributos iguales) y
quedan registradas aparte. Los nulos y demás anomalías **se conservan** y se marcan en `observacion_calidad`.

## 0. Entorno de ejecución

In [ ]:
# Entorno: Google Colab + Google Drive (única herramienta del proyecto)
import os, sys
try:
    import google.colab  # noqa: F401
except ImportError as e:
    raise RuntimeError('Este notebook se ejecuta en Google Colab.') from e

if not os.path.ismount('/content/drive'):
    from google.colab import drive
    drive.mount('/content/drive')  # Colab pide autorizar el acceso a Drive

BASE = '/content/drive/MyDrive/BigData_U2'   # carpeta ÚNICA del proyecto en Drive (datos, informes, evidencia)
LIB = '/content/u2lib'                         # código de apoyo que escribe este notebook (disco del VM; no se sube)
os.makedirs(BASE, exist_ok=True)
for carpeta in ('config', 'src/emergencias123/reportes', 'tests'):
    os.makedirs(f'{LIB}/{carpeta}', exist_ok=True)
sys.path.insert(0, f'{LIB}/src')
os.environ['U2_ENTORNO'] = 'colab'
print('Carpeta del proyecto en Drive:', BASE)

## Anexo A · Código de apoyo (ejecutar primero)

Estas celdas **escriben en el disco del VM** (`/content/u2lib`) el código de apoyo que usa el resto del notebook: la
configuración, las reglas de normalización, las operaciones de cada herramienta, el pipeline, las validaciones y
las pruebas. Cada archivo lleva antes una nota de qué hace. **No hace falta leerlas para seguir el relato**: el
código que importa se muestra en pantalla en las secciones donde se explica. Solo hay que ejecutarlas antes (la
opción *Ejecutar todas* ya lo hace). Nada de esto se sube a Drive.

**Configuración (`config/config.yaml`).** Toda ruta y todo parámetro sale de aquí; el código no tiene rutas fijas.
Contiene las rutas de Drive, los 3 archivos con su periodo, su **huella SHA-256** y la URL del portal (respaldo
heredado de la Unidad 1), el esquema observado en la inspección, los parámetros del benchmark y del PoC, la
configuración de Spark y los datos del bloque de título APA de los informes.

In [ ]:
%%writefile /content/u2lib/config/config.yaml
# Configuración del proyecto U2 — Emergencias 123 Bogotá.
# Las rutas relativas se resuelven contra la raíz del código (carpeta que contiene config/).
# El entorno se detecta solo (Colab si existe /content/drive/MyDrive); se puede forzar con U2_ENTORNO.

entornos:
  colab:                          # flujo principal: todo ocurre en Colab; una sola carpeta en Drive (BigData_U2)
    base: /content/drive/MyDrive/BigData_U2
    datos: /content/drive/MyDrive/BigData_U2/datos
    resultados: /content/drive/MyDrive/BigData_U2/resultados
    informes: /content/u2_informes   # temporal en el VM: el notebook de informes copia cada archivo a su Actividad_N/
    datos_alternos:               # si faltan CSV en `datos`: carpeta de Drive de la Unidad 1 (guía U1, paso 2)
      - /content/drive/MyDrive/BigData_Emergencias_123
  local:                          # solo para desarrollo y pruebas del código
    base: .
    datos: BigData_Emergencias_123
    resultados: resultados
    informes: informes

datos:
  separador: ";"
  codificacion: utf-8-sig        # los CSV traen BOM UTF-8
  # Fuente: Secretaría Distrital de Salud (SDS), portal Datos Abiertos Bogotá, licencia CC BY 4.0.
  # sha256 = huella de los archivos usados (idénticos a los del portal, verificado el 2026-11-10); `url` es el
  # respaldo de descarga que ya usaba la Unidad 1 cuando Drive no tiene los CSV.
  conjunto_url: https://datosabiertos.bogota.gov.co/dataset/llamadas-de-urgencias-y-emergencias-que-ingresan-a-traves-de-la-linea-123
  archivos:                      # periodo = mes que publica cada archivo
    - nombre: llamadas123_julio_2026.csv
      periodo: "2026-07"
      sha256: 21cbe766616d324086a26d91b337cbc3bf89e42585f2ffd0efb218d574b27589
      url: https://datosabiertos.bogota.gov.co/dataset/b99881e0-8750-4fa8-af6c-3f30c9e38708/resource/e2b98357-3305-4058-a113-982490ab6b7e/download/llamadas123_julio_2026.csv
    - nombre: llamadaslinea123agosto2026.csv
      periodo: "2026-08"
      sha256: c421fe9d75edc4ed6da2afa153fe6c51e6576f371b4d140046aec57b66e5b2e1
      url: https://datosabiertos.bogota.gov.co/dataset/b99881e0-8750-4fa8-af6c-3f30c9e38708/resource/a79acb9e-57ae-4e6c-bc15-3a8687330b1f/download/llamadaslinea123agosto2026.csv
    - nombre: llamadaslinea123Septiembre.csv
      periodo: "2026-09"
      sha256: face37df8c67912796511f9e6392d2a12dc139c91360eac1dfa46490aedd32d2
      url: https://datosabiertos.bogota.gov.co/dataset/b99881e0-8750-4fa8-af6c-3f30c9e38708/resource/39f0e478-08f0-405c-b807-8015e07baec1/download/llamadaslinea123.csv

esquema:
  columnas:
    - NUMERO_INCIDENTE
    - FECHA_INICIO_DESPLAZAMIENTO_MOVIL
    - CODIGO_LOCALIDAD
    - LOCALIDAD
    - EDAD
    - UNIDAD
    - GENERO
    - TIPO_INCIDENTE
    - PRIORIDAD_FINAL
    - RECEPCION
  criticas:                      # un nulo aquí => el registro se conserva con observación de calidad
    - NUMERO_INCIDENTE
    - FECHA_INICIO_DESPLAZAMIENTO_MOVIL
    - CODIGO_LOCALIDAD
    - LOCALIDAD
    - TIPO_INCIDENTE
  valores_nulos: ["N/A", ""]     # marcadores de ausencia observados en la inspección
  localidad_sin_dato: SIN_D
  formato_fecha: "%Y-%m-%d %H:%M:%S"
  formato_fecha_spark: "yyyy-MM-dd HH:mm:ss"

catalogos:
  tipos: config/catalogo_tipos_derivado.csv          # derivado de los datos, NO oficial
  localidades: config/localidades_bogota.csv         # división político-administrativa de Bogotá

benchmark:
  herramientas: [pandas, polars, spark]
  modos: [por_operacion, extremo_a_extremo, extremo_a_extremo_caliente]   # el último es complementario
  localidades_filtro: [USAQUEN, CHAPINERO]           # valores ya normalizados (sin tildes)
  calentamientos: 1
  repeticiones: 5
  intervalo_muestreo_s: 0.02

poc:
  ejecuciones: 3
  tolerancia_dias_fin_mes: 1     # registros del día 1 del mes siguiente siguen en la ventana del archivo
  edad_min_anios: 0
  edad_max_anios: 120
  particionamiento: periodo      # periodo | periodo_localidad
  factores_edad:                 # conversión de EDAD a años según UNIDAD (normalizada)
    ANOS: 1.0
    MESES: 0.0833333333
    DIAS: 0.0027378508
    HORAS: 0.0001140771

documentos:                      # datos del bloque de título APA (completar antes de entregar)
  autor: "[Nombre del estudiante o del equipo]"
  institucion: "[Nombre de la institución]"
  curso: "Herramientas de Big Data — Saber 2: Procesamiento distribuido"
  docente: "[Nombre del docente]"
  fuente_datos: "Secretaría Distrital de Salud de Bogotá"           # autor (organización publicadora en el portal)
  titulo_datos: "Llamadas de urgencias y emergencias que ingresan a través de la línea 123"
  anio_datos: "2026"                                                # recursos jul–sep publicados en 2026
  editor_datos: "Datos Abiertos Bogotá"

spark:
  app_name: PoC_Emergencias123_Bogota
  driver_memory_max_gb: 4        # valor de la guía; se limita al 50 % de la RAM detectada
  driver_max_result_size: 2g
  advisory_partition_size: 64MB
  shuffle_partitions_por_cpu: 2
  zona_horaria: America/Bogota

Paquete `emergencias123`, que agrupa el código de apoyo de los notebooks.

In [ ]:
%%writefile /content/u2lib/src/emergencias123/__init__.py
"""Proyecto U2 — Emergencias 123 Bogotá: benchmark Pandas/Polars/PySpark y PoC en PySpark."""

__version__ = '1.0.0'

Subpaquete `reportes` (gráficos e informes).

In [ ]:
%%writefile /content/u2lib/src/emergencias123/reportes/__init__.py
"""Generación de gráficos e informes a partir de las mediciones guardadas."""

**Lectura de la configuración (`config.py`).** Carga el YAML, resuelve las rutas del entorno de Colab y ofrece
utilidades (`rutas_csv`, `dir_salida`). Las variables `U2_DIR_DATOS` y `U2_DIR_RESULTADOS` permiten que el
benchmark trabaje sobre una copia en el disco local del VM sin cambiar el código.

In [ ]:
%%writefile /content/u2lib/src/emergencias123/config.py
"""Carga de config/config.yaml y resolución de rutas según el entorno (local o Colab)."""
import os
from dataclasses import dataclass
from pathlib import Path

import yaml

RAIZ_CODIGO = Path(__file__).resolve().parents[2]
RUTA_CONFIG = RAIZ_CODIGO / 'config' / 'config.yaml'


def detectar_entorno():
    forzado = os.environ.get('U2_ENTORNO')
    if forzado:
        return forzado
    return 'colab' if Path('/content/drive/MyDrive').exists() else 'local'


@dataclass(frozen=True)
class Config:
    datos: dict
    esquema: dict
    catalogos: dict
    benchmark: dict
    poc: dict
    spark: dict
    documentos: dict
    entorno: str
    dir_datos: Path
    dir_resultados: Path
    dir_informes: Path
    dir_base: Path
    datos_alternos: tuple = ()     # otras carpetas donde buscar los CSV (p. ej. la de Drive de la Unidad 1)

    def ruta_csv(self, nombre):
        return self.dir_datos / nombre

    @property
    def rutas_csv(self):
        return [self.ruta_csv(a['nombre']) for a in self.datos['archivos']]

    @property
    def periodo_por_archivo(self):
        return {a['nombre']: a['periodo'] for a in self.datos['archivos']}

    @property
    def ruta_catalogo_tipos(self):
        return RAIZ_CODIGO / self.catalogos['tipos']

    @property
    def ruta_localidades(self):
        return RAIZ_CODIGO / self.catalogos['localidades']

    def dir_salida(self, *partes):
        ruta = self.dir_resultados.joinpath(*partes)
        ruta.mkdir(parents=True, exist_ok=True)
        return ruta


def _resolver(ruta):
    ruta = Path(ruta)
    return ruta if ruta.is_absolute() else RAIZ_CODIGO / ruta


def cargar_config(ruta=None, entorno=None, dir_resultados=None):
    """Lee el YAML. `dir_resultados` permite aislar salidas (pruebas o ejecuciones)."""
    with open(ruta or RUTA_CONFIG, encoding='utf-8') as f:
        crudo = yaml.safe_load(f)
    entorno = entorno or detectar_entorno()
    rutas = crudo['entornos'][entorno]
    return Config(
        datos=crudo['datos'], esquema=crudo['esquema'], catalogos=crudo['catalogos'],
        benchmark=crudo['benchmark'], poc=crudo['poc'], spark=crudo['spark'], documentos=crudo.get('documentos', {}),
        entorno=entorno,
        dir_datos=_resolver(os.environ.get('U2_DIR_DATOS', rutas['datos'])),
        dir_resultados=_resolver(dir_resultados or os.environ.get('U2_DIR_RESULTADOS', rutas['resultados'])),
        dir_informes=_resolver(rutas['informes']),
        dir_base=_resolver(rutas['base']),
        datos_alternos=tuple(_resolver(r) for r in rutas.get('datos_alternos', [])),
    )

**Tabla de localidades (`localidades_bogota.csv`).** División político-administrativa oficial de Bogotá
(códigos 1–20). Sirve para validar que el código y el nombre de localidad de cada registro sean coherentes.

In [ ]:
%%writefile /content/u2lib/config/localidades_bogota.csv
codigo_localidad,localidad_norm,localidad_oficial
1,USAQUEN,Usaquén
2,CHAPINERO,Chapinero
3,SANTA FE,Santa Fe
4,SAN CRISTOBAL,San Cristóbal
5,USME,Usme
6,TUNJUELITO,Tunjuelito
7,BOSA,Bosa
8,KENNEDY,Kennedy
9,FONTIBON,Fontibón
10,ENGATIVA,Engativá
11,SUBA,Suba
12,BARRIOS UNIDOS,Barrios Unidos
13,TEUSAQUILLO,Teusaquillo
14,LOS MARTIRES,Los Mártires
15,ANTONIO NARINO,Antonio Nariño
16,PUENTE ARANDA,Puente Aranda
17,LA CANDELARIA,La Candelaria
18,RAFAEL URIBE URIBE,Rafael Uribe Uribe
19,CIUDAD BOLIVAR,Ciudad Bolívar
20,SUMAPAZ,Sumapaz

**Catálogo de tipos de incidente (`catalogo_tipos_derivado.csv`) — derivado de los datos, NO oficial.** No se
encontró una tabla oficial de códigos entre los insumos, así que el catálogo se derivó de los propios CSV
(23 códigos: 19 con la forma `COD - DESCRIPCIÓN` y 4 sin código). En la parte 2 se vuelve a derivar desde los
datos y se comprueba que coincide con esta tabla.

In [ ]:
%%writefile /content/u2lib/config/catalogo_tipos_derivado.csv
codigo_tipo,descripcion_tipo,codigo_explicito,fuente
ACCAVI,ACCIDENTE DE AVIACIÓN,True,Derivado de TIPO_INCIDENTE de los CSV jul-sep 2026 (no oficial)
ACOMPANAMIENTO A EVENTO,ACOMPAÑAMIENTO A EVENTO,False,Derivado de TIPO_INCIDENTE de los CSV jul-sep 2026 (no oficial)
ACV,ACCIDENTE CEREBRO VASCULAR,True,Derivado de TIPO_INCIDENTE de los CSV jul-sep 2026 (no oficial)
AMESUI,AMENAZA DE SUICIDIO,True,Derivado de TIPO_INCIDENTE de los CSV jul-sep 2026 (no oficial)
APOYO CRUE,APOYO CRUE,False,Derivado de TIPO_INCIDENTE de los CSV jul-sep 2026 (no oficial)
CAIALT,CAÍDA DE ALTURA,True,Derivado de TIPO_INCIDENTE de los CSV jul-sep 2026 (no oficial)
CONVULSION,CONVULSIÓN,True,Derivado de TIPO_INCIDENTE de los CSV jul-sep 2026 (no oficial)
DOLTOR,DOLOR TORÁCICO,True,Derivado de TIPO_INCIDENTE de los CSV jul-sep 2026 (no oficial)
ELECTROCUC,ELECTROCUCIÓN / RESCATE,True,Derivado de TIPO_INCIDENTE de los CSV jul-sep 2026 (no oficial)
ENFERMO,ENFERMO,False,Derivado de TIPO_INCIDENTE de los CSV jul-sep 2026 (no oficial)
EVENTO NATURAL,EVENTO NATURAL,False,Derivado de TIPO_INCIDENTE de los CSV jul-sep 2026 (no oficial)
EVERES,EVENTO RESPIRATORIO,True,Derivado de TIPO_INCIDENTE de los CSV jul-sep 2026 (no oficial)
HERIDO,HERIDOS ACCIDENTALES,True,Derivado de TIPO_INCIDENTE de los CSV jul-sep 2026 (no oficial)
INCONSCIEN,INCONSCIENTE O PARO CARDIORRESPIRATORIO,True,Derivado de TIPO_INCIDENTE de los CSV jul-sep 2026 (no oficial)
INTOX,INTOXICACIÓN,True,Derivado de TIPO_INCIDENTE de los CSV jul-sep 2026 (no oficial)
INTSUI,INTENTO DE SUICIDIO,True,Derivado de TIPO_INCIDENTE de los CSV jul-sep 2026 (no oficial)
MALTRATO,MALTRATO,True,Derivado de TIPO_INCIDENTE de los CSV jul-sep 2026 (no oficial)
PATGIN,PATOLOGÍA GINECOBSTÉTRICA,True,Derivado de TIPO_INCIDENTE de los CSV jul-sep 2026 (no oficial)
QUEMADURAS,QUEMADURAS,True,Derivado de TIPO_INCIDENTE de los CSV jul-sep 2026 (no oficial)
SANVAG,SANGRADO VAGINAL,True,Derivado de TIPO_INCIDENTE de los CSV jul-sep 2026 (no oficial)
SINTOGASTR,SÍNTOMAS GASTROINTESTINALES,True,Derivado de TIPO_INCIDENTE de los CSV jul-sep 2026 (no oficial)
TRASTMENT,TRASTORNO MENTAL,True,Derivado de TIPO_INCIDENTE de los CSV jul-sep 2026 (no oficial)
VIOSEXUAL,VIOLENCIA SEXUAL,True,Derivado de TIPO_INCIDENTE de los CSV jul-sep 2026 (no oficial)

**Normalización de texto (`normalizacion.py`).** Reglas compartidas por las tres herramientas: quitar tildes,
colapsar espacios y pasar a mayúsculas (`USAQUÉN` → `USAQUEN`), y separar el código del tipo de incidente de su
descripción. Al definirlas una vez se garantiza que Pandas, Polars y PySpark apliquen la misma lógica.

In [ ]:
%%writefile /content/u2lib/src/emergencias123/normalizacion.py
"""Reglas de normalización de texto compartidas por las tres herramientas.

Se usa un mapa explícito de tildes del español (y no NFKD) porque Spark no tiene una función
de normalización Unicode. Así Pandas, Polars y Spark aplican exactamente la misma transformación.
La inspección de los datos confirmó que no hay otros caracteres no ASCII en los campos normalizados.
"""
import re

CON_TILDE = 'ÁÀÄÂÉÈËÊÍÌÏÎÓÒÖÔÚÙÜÛÑáàäâéèëêíìïîóòöôúùüûñ'
SIN_TILDE = 'AAAAEEEEIIIIOOOOUUUUNaaaaeeeeiiiioooouuuun'
TABLA_TILDES = str.maketrans(CON_TILDE, SIN_TILDE)
SEPARADOR_CODIGO = ' - '
ESPACIOS = re.compile(r'\s+')


def normalizar_texto(valor):
    """Quita tildes, colapsa espacios, recorta y pasa a mayúsculas. Vacío -> None."""
    if valor is None:
        return None
    texto = ESPACIOS.sub(' ', str(valor).translate(TABLA_TILDES)).strip().upper()
    return texto or None


def colapsar_espacios(valor):
    if valor is None:
        return None
    return ESPACIOS.sub(' ', str(valor)).strip() or None


def extraer_codigo_tipo(tipo_incidente):
    """'INTSUI - INTENTO DE SUICIDIO' -> 'INTSUI'; sin código ('ENFERMO') -> el texto completo."""
    texto = normalizar_texto(tipo_incidente)
    if texto is None:
        return None
    return texto.split(SEPARADOR_CODIGO, 1)[0]


def extraer_descripcion_tipo(tipo_incidente):
    """Descripción legible (con tildes): lo que sigue al primer ' - ', o el texto completo."""
    texto = colapsar_espacios(tipo_incidente)
    if texto is None:
        return None
    return texto.split(SEPARADOR_CODIGO, 1)[-1]

**Adquisición de los datos (`adquisicion.py`), adaptada de la Unidad 1.** La Unidad 1 leía los CSV de
`MyDrive/BigData_Emergencias_123` y, si Drive no estaba disponible, los descargaba del portal con reintentos.
Al revisar esa lógica se mantuvieron sus dos fuentes y los reintentos, y se corrigieron dos puntos: el fallo del
montaje de Drive ya no se ignora en silencio (aquí Drive es obligatorio, porque los resultados se guardan allí), y
todo archivo se verifica con su **SHA-256** antes de usarse, porque el portal podría reemplazar un recurso.

In [ ]:
%%writefile /content/u2lib/src/emergencias123/adquisicion.py
"""Garantiza que los 3 CSV estén en la carpeta de datos del proyecto, íntegros y trazables.

Adaptado de la celda de configuración de la Unidad 1 (copia en U1_referencia/), que leía los CSV de
`MyDrive/BigData_Emergencias_123` y, si Drive no estaba disponible, los descargaba del portal Datos Abiertos
Bogotá con reintentos. Cambios respecto a U1, tras revisar su lógica:
- U1 ignoraba en silencio un fallo de `drive.mount` (`except Exception: pass`). En U2 Drive es obligatorio porque
  los resultados (Parquet, mediciones, informes) se escriben allí; el montaje lo resuelve el notebook.
- U1 no comprobaba qué descargaba. Aquí cada archivo, venga de donde venga, se verifica con el SHA-256 de
  `config.yaml`; si no coincide, se detiene con un error y no se usa (el portal podría reemplazar un recurso).
- El origen de cada archivo queda registrado (local, carpeta U1 o portal) para la trazabilidad.
Orden de búsqueda: carpeta `datos` del proyecto → carpetas alternas (por nombre o por SHA-256) → portal.
"""
import hashlib
import logging
import shutil
import time
import urllib.request

log = logging.getLogger(__name__)


def sha256(ruta):
    h = hashlib.sha256()
    with open(ruta, 'rb') as f:
        for bloque in iter(lambda: f.read(1 << 20), b''):
            h.update(bloque)
    return h.hexdigest()


def _descargar(url, destino, intentos=5, espera_s=10, timeout_s=90):
    """Descarga con reintentos (el portal a veces corta el handshake TLS, observado el 2026-11-10)."""
    ultimo = None
    for i in range(1, intentos + 1):
        try:
            solicitud = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
            with urllib.request.urlopen(solicitud, timeout=timeout_s) as r:
                destino.write_bytes(r.read())
            return
        except OSError as e:
            ultimo = e
            log.warning('Descarga fallida (%s/%s) %s: %s', i, intentos, url, e)
            time.sleep(espera_s)
    raise RuntimeError(f'No se pudo descargar {url} tras {intentos} intentos: {ultimo}')


def _buscar_alterno(carpetas, nombre, esperado):
    """Busca el archivo por nombre y, si no, por contenido: la carpeta de la Unidad 1 los nombra distinto
    (p. ej. `llamadaslinea123_agosto_2026.csv`), así que se acepta cualquier CSV con el SHA-256 esperado."""
    for d in carpetas:
        if not d.is_dir():
            continue
        if (d / nombre).exists() and (not esperado or sha256(d / nombre) == esperado):
            return d / nombre
        if esperado:
            for c in sorted(d.glob('*.csv')):
                if sha256(c) == esperado:
                    return c
    return None


def asegurar_datos(cfg, permitir_descarga=True):
    """Deja los CSV en `cfg.dir_datos` con la huella esperada. Devuelve {archivo: {origen, sha256}}.

    Nunca sobrescribe un archivo existente con huella distinta: eso indicaría datos diferentes a los
    analizados y se informa como error en lugar de corregirse en silencio.
    """
    cfg.dir_datos.mkdir(parents=True, exist_ok=True)
    resultado = {}
    for a in cfg.datos['archivos']:
        nombre, esperado = a['nombre'], a.get('sha256')
        destino = cfg.dir_datos / nombre
        if destino.exists():
            origen = 'carpeta del proyecto'
        else:
            alterno = _buscar_alterno(cfg.datos_alternos, nombre, esperado)
            if alterno is not None:
                shutil.copy2(alterno, destino)
                origen = f'copiado de {alterno.parent}'
            elif permitir_descarga and a.get('url'):
                temporal = destino.with_suffix('.descarga')
                _descargar(a['url'], temporal)
                if esperado and sha256(temporal) != esperado:
                    temporal.unlink()
                    raise RuntimeError(f'{nombre} descargado del portal no coincide con el SHA-256 esperado: '
                                       'el recurso cambió y no se usa.')
                temporal.replace(destino)
                origen = 'descargado del portal Datos Abiertos Bogotá'
            else:
                raise FileNotFoundError(f'Falta {nombre} en {cfg.dir_datos} y no hay respaldo disponible.')
        huella = sha256(destino)
        if esperado and huella != esperado:
            raise RuntimeError(f'{destino} no coincide con el SHA-256 esperado ({huella[:16]}… ≠ {esperado[:16]}…).')
        resultado[nombre] = {'origen': origen, 'sha256': huella}
    return resultado

**Catálogo derivado (`catalogo.py`).** Lee los valores de `TIPO_INCIDENTE`, separa código y descripción y
construye la tabla de códigos únicos. Siempre se rotula como **derivado de los datos, no oficial**.

In [ ]:
%%writefile /content/u2lib/src/emergencias123/catalogo.py
"""Catálogo de tipos de incidente DERIVADO de los propios datos (no es un catálogo oficial).

No se encontró una tabla oficial de códigos de emergencia entre los insumos de U2. El catálogo
se construye una sola vez a partir de TIPO_INCIDENTE (`COD - DESCRIPCIÓN`, o solo la descripción
cuando no hay código) y se versiona en config/ como insumo externo del pipeline.
"""
import csv
from collections import defaultdict

from .normalizacion import SEPARADOR_CODIGO, colapsar_espacios, extraer_codigo_tipo, extraer_descripcion_tipo

FUENTE = 'Derivado de TIPO_INCIDENTE de los CSV jul-sep 2026 (no oficial)'
COLUMNAS = ['codigo_tipo', 'descripcion_tipo', 'codigo_explicito', 'fuente']


def leer_valores(rutas, columna, separador, codificacion):
    conteos = defaultdict(int)
    for ruta in rutas:
        with open(ruta, encoding=codificacion, newline='') as f:
            for fila in csv.DictReader(f, delimiter=separador):
                conteos[fila[columna]] += 1
    return dict(conteos)


def derivar_catalogo(valores_tipo):
    """Devuelve las filas del catálogo. Falla si un código queda asociado a dos descripciones."""
    por_codigo = {}
    for valor in sorted(v for v in valores_tipo if colapsar_espacios(v)):
        codigo = extraer_codigo_tipo(valor)
        fila = {
            'codigo_tipo': codigo,
            'descripcion_tipo': extraer_descripcion_tipo(valor),
            'codigo_explicito': SEPARADOR_CODIGO in colapsar_espacios(valor),
            'fuente': FUENTE,
        }
        previa = por_codigo.get(codigo)
        if previa and previa['descripcion_tipo'] != fila['descripcion_tipo']:
            raise ValueError(f'Código ambiguo {codigo}: {previa["descripcion_tipo"]!r} vs {fila["descripcion_tipo"]!r}')
        por_codigo[codigo] = fila
    return [por_codigo[c] for c in sorted(por_codigo)]


def guardar_catalogo(filas, ruta):
    with open(ruta, 'w', encoding='utf-8', newline='') as f:
        escritor = csv.DictWriter(f, fieldnames=COLUMNAS)
        escritor.writeheader()
        escritor.writerows(filas)


def leer_catalogo(ruta):
    with open(ruta, encoding='utf-8', newline='') as f:
        filas = list(csv.DictReader(f))
    for fila in filas:
        fila['codigo_explicito'] = fila['codigo_explicito'] == 'True'
    codigos = [f['codigo_tipo'] for f in filas]
    if len(codigos) != len(set(codigos)):
        raise ValueError('La clave codigo_tipo del catálogo no es única')
    return filas

**Inventario de datos (`inventario.py`).** Procedencia, tamaño, huella, esquema, nulos, rangos de fechas,
registros fuera del mes de su archivo y duplicados exactos de cada CSV.

In [ ]:
%%writefile /content/u2lib/src/emergencias123/inventario.py
"""Inventario de los CSV de entrada: procedencia, tamaño, huella, esquema, nulos y rangos.

Usa solo la biblioteca estándar para que el perfil no dependa de ninguna de las herramientas
que se comparan en el benchmark.
"""
import csv
import hashlib
from collections import Counter
from datetime import datetime, timedelta

from .normalizacion import normalizar_texto


def _sha256(ruta):
    h = hashlib.sha256()
    with open(ruta, 'rb') as f:
        for bloque in iter(lambda: f.read(1 << 20), b''):
            h.update(bloque)
    return h.hexdigest()


def _fin_de_mes(periodo):
    anio, mes = map(int, periodo.split('-'))
    return datetime(anio + (mes == 12), mes % 12 + 1, 1)


def perfilar_archivo(ruta, periodo, cfg):
    esq = cfg.esquema
    nulos_marcados = set(esq['valores_nulos'])
    with open(ruta, 'rb') as f:
        crudo = f.read()
    lineas_fisicas = crudo.count(b'\n') + (0 if crudo.endswith(b'\n') else 1)
    with open(ruta, encoding=cfg.datos['codificacion'], newline='') as f:
        lector = csv.DictReader(f, delimiter=cfg.datos['separador'])
        columnas = lector.fieldnames
        filas = list(lector)
    nulos = {c: sum(1 for r in filas if (r[c] or '').strip() in nulos_marcados) for c in columnas}
    fechas, invalidas = [], 0
    for r in filas:
        try:
            fechas.append(datetime.strptime(r['FECHA_INICIO_DESPLAZAMIENTO_MOVIL'], esq['formato_fecha']))
        except (ValueError, TypeError):
            invalidas += 1
    inicio = datetime.strptime(periodo + '-01', '%Y-%m-%d')
    fin = _fin_de_mes(periodo)
    fuera_mes = sum(1 for d in fechas if not inicio <= d < fin)
    fuera_ventana = sum(1 for d in fechas if not inicio <= d < fin + timedelta(days=cfg.poc['tolerancia_dias_fin_mes']))
    claves = [tuple(r[c] for c in columnas) for r in filas]
    return {
        'archivo': ruta.name,
        'periodo': periodo,
        'bytes': len(crudo),
        'sha256': _sha256(ruta),
        'bom_utf8': crudo.startswith(b'\xef\xbb\xbf'),
        'fin_de_linea': 'CRLF' if b'\r\n' in crudo[:4096] else 'LF',
        'lineas_fisicas': lineas_fisicas,
        'registros': len(filas),
        'columnas': columnas,
        'nulos_o_na': nulos,
        'fechas_invalidas': invalidas,
        'fecha_min': min(fechas).isoformat(sep=' ') if fechas else None,
        'fecha_max': max(fechas).isoformat(sep=' ') if fechas else None,
        'fuera_del_mes_del_archivo': fuera_mes,
        'fuera_de_ventana_con_tolerancia': fuera_ventana,
        'duplicados_exactos': len(claves) - len(set(claves)),
        'localidades': dict(Counter(r['LOCALIDAD'] for r in filas).most_common()),
        'tipos_incidente': dict(Counter(r['TIPO_INCIDENTE'] for r in filas).most_common()),
    }


def perfilar(cfg):
    archivos = [perfilar_archivo(cfg.ruta_csv(a['nombre']), a['periodo'], cfg) for a in cfg.datos['archivos']]
    todas = []
    for a in cfg.datos['archivos']:
        with open(cfg.ruta_csv(a['nombre']), encoding=cfg.datos['codificacion'], newline='') as f:
            todas.extend(csv.DictReader(f, delimiter=cfg.datos['separador']))
    ids = Counter(r['NUMERO_INCIDENTE'] for r in todas)
    claves = [tuple(r.values()) for r in todas]
    pares = Counter((r['CODIGO_LOCALIDAD'], normalizar_texto(r['LOCALIDAD'])) for r in todas)
    return {
        'archivos': archivos,
        'total_registros': len(todas),
        'duplicados_exactos_global': len(claves) - len(set(claves)),
        'filas_con_incidente_repetido': sum(n for n in ids.values() if n > 1),
        'incidentes_unicos': len(ids),
        'pares_codigo_localidad': {f'{c}|{n}': k for (c, n), k in sorted(pares.items())},
    }


def a_markdown(inv, catalogo, entorno_local):
    """Documento docs/inventario_datos.md a partir del inventario medido."""
    a0 = inv['archivos'][0]
    l = ['# Inventario de datos — Emergencias 123 Bogotá (U2)', '',
         'Generado en Google Colab por el notebook de procesamiento (`emergencias123.inventario`) a partir de los CSV en Drive, sin modificarlos.', '',
         '## Procedencia', '',
         '- Conjunto «Llamadas de urgencias y emergencias que ingresan a través de la línea 123», publicado por la',
         '  Secretaría Distrital de Salud en el portal Datos Abiertos Bogotá (licencia CC BY 4.0); un archivo por mes.',
         '- Los archivos se toman de la carpeta de Drive de la Unidad 1 o se descargan del portal; la integridad se fija',
         '  con la huella SHA-256 de cada archivo (idénticos a los del portal, verificado el 2026-11-10).',
         '- Meses seleccionados: julio, agosto y septiembre de 2026.', '',
         '## Archivos', '',
         '| Archivo | Periodo | Bytes | Registros | Líneas físicas | Fecha mín. | Fecha máx. | Fuera del mes | Dup. exactos | SHA-256 |',
         '|---|---|---:|---:|---:|---|---|---:|---:|---|']
    for a in inv['archivos']:
        l.append(f"| {a['archivo']} | {a['periodo']} | {a['bytes']:,} | {a['registros']:,} | {a['lineas_fisicas']:,} | "
                 f"{a['fecha_min']} | {a['fecha_max']} | {a['fuera_del_mes_del_archivo']} | {a['duplicados_exactos']} | "
                 f"`{a['sha256'][:16]}…` |")
    l += ['', f"**Total:** {inv['total_registros']:,} registros · {inv['incidentes_unicos']:,} incidentes únicos · "
              f"{inv['filas_con_incidente_repetido']:,} filas con NUMERO_INCIDENTE repetido · "
              f"{inv['duplicados_exactos_global']} copias duplicadas exactas.", '',
          '## Formato y esquema', '',
          f"- Codificación UTF-8 con BOM: {a0['bom_utf8']}; fin de línea {a0['fin_de_linea']}; separador `;`; encabezado en la primera línea.",
          '- Columnas (todas se leen como texto y se tipan después):', '']
    l += [f'  {i + 1}. `{c}` — nulos/`N/A`: {sum(a["nulos_o_na"][c] for a in inv["archivos"]):,}'
          for i, c in enumerate(a0['columnas'])]
    l += ['', '## Hallazgos que condicionan el diseño', '',
          '- `EDAD`, `UNIDAD` y `GENERO` usan `N/A` como marcador de ausencia: se convierte a nulo y no se imputa.',
          '- `RECEPCION` vacía significa que no hay recepción registrada: es un nulo legítimo, no un error.',
          '- Cada archivo incluye registros de la madrugada del día 1 del mes siguiente '
          f"({', '.join(str(a['fuera_del_mes_del_archivo']) for a in inv['archivos'])} registros). "
          'Se conservan con el periodo de su archivo y se valida una ventana de 1 día de tolerancia.',
          '- Las localidades vienen con tilde (`USAQUÉN`) y se normalizan a `USAQUEN`. Hay 1 fila con `SIN_D` '
          '(localidad sin dato), que el PoC conserva marcada en `observacion_calidad`.',
          '- `NUMERO_INCIDENTE` se repite cuando un incidente tiene varios despachos: no es un duplicado.',
          '- Las copias duplicadas exactas (100 % de los atributos iguales) se eliminan en el PoC y se guardan aparte; '
          'los nulos no se eliminan ni se imputan.', '',
          '## Catálogo de tipos de incidente (derivado, NO oficial)', '',
          'No se encontró una tabla oficial de códigos de emergencia entre los insumos. '
          '`config/catalogo_tipos_derivado.csv` se derivó de `TIPO_INCIDENTE` '
          f"({len(catalogo)} códigos; {sum(c['codigo_explicito'] for c in catalogo)} con código explícito "
          f"`COD - DESCRIPCIÓN` y {sum(not c['codigo_explicito'] for c in catalogo)} sin código, en los que "
          'el texto normalizado actúa como código). Para usar un catálogo oficial basta con reemplazar ese archivo '
          'conservando sus columnas.', '',
          '## Pares código ↔ localidad observados', '',
          '| Código | Localidad normalizada | Registros |', '|---|---|---:|']
    l += [f'| {k.split("|")[0]} | {k.split("|")[1]} | {v:,} |' for k, v in inv['pares_codigo_localidad'].items()]
    l += ['', '## Entorno de ejecución', '', *[f'- {k}: {v}' for k, v in entorno_local.items()], '']
    return '\n'.join(l)

**Muestreo de memoria (`memoria.py`).** Un hilo registra cada 20 ms el % de memoria del sistema y el RSS del
proceso y sus hijos; el pico es el máximo de las muestras, no una lectura final.

In [ ]:
%%writefile /content/u2lib/src/emergencias123/memoria.py
"""Muestreo de memoria durante la ejecución (el pico es el máximo de las muestras, no una lectura final).

Se registran dos magnitudes distintas:
- `mem_sis_pct`: psutil.virtual_memory().percent, el % de RAM ocupada en TODO el sistema
  (lo que pide la guía). Incluye otros procesos de Colab, así que no es memoria exclusiva de la herramienta.
- `rss_mb`: RSS del proceso Python más sus hijos recursivos (incluye la JVM de Spark, lanzada por py4j).
  RSS cuenta páginas compartidas, así que la suma puede sobrestimar algo la memoria privada.
Limitación: un pico más corto que el intervalo de muestreo puede no quedar registrado.
"""
import threading
import time

import psutil


def rss_arbol_mb(proceso):
    total = 0
    for p in [proceso, *proceso.children(recursive=True)]:
        try:
            total += p.memory_info().rss
        except (psutil.NoSuchProcess, psutil.AccessDenied):
            pass
    return total / 2 ** 20


class MemorySampler:
    """Hilo que muestrea memoria cada `intervalo` segundos y lleva el pico del tramo en curso."""

    def __init__(self, intervalo=0.02):
        self.intervalo = intervalo
        self._proceso = psutil.Process()
        self._bloqueo = threading.Lock()
        self._parar = threading.Event()
        self._hilo = threading.Thread(target=self._bucle, daemon=True)
        self._tramo = None

    def _muestra(self):
        return psutil.virtual_memory().percent, rss_arbol_mb(self._proceso)

    def _registrar(self, sis, rss):
        with self._bloqueo:
            if self._tramo is not None:
                t = self._tramo
                t['mem_sis_pico_pct'] = max(t['mem_sis_pico_pct'], sis)
                t['rss_pico_mb'] = max(t['rss_pico_mb'], rss)
                t['n_muestras'] += 1

    def _bucle(self):
        while not self._parar.is_set():
            self._registrar(*self._muestra())
            self._parar.wait(self.intervalo)

    def __enter__(self):
        self._hilo.start()
        return self

    def __exit__(self, *exc):
        self._parar.set()
        self._hilo.join()

    def iniciar_tramo(self):
        sis, rss = self._muestra()
        with self._bloqueo:
            self._tramo = {'mem_sis_base_pct': sis, 'mem_sis_pico_pct': sis,
                           'rss_base_mb': rss, 'rss_pico_mb': rss, 'n_muestras': 1}

    def cerrar_tramo(self):
        self._registrar(*self._muestra())
        with self._bloqueo:
            tramo, self._tramo = self._tramo, None
        return {k: round(v, 3) if isinstance(v, float) else v for k, v in tramo.items()}


def medir(sampler, funcion, *args, **kwargs):
    """Ejecuta `funcion` midiendo tiempo (reloj monotónico) y memoria muestreada."""
    sampler.iniciar_tramo()
    t0 = time.perf_counter_ns()
    resultado = funcion(*args, **kwargs)
    segundos = (time.perf_counter_ns() - t0) / 1e9
    return resultado, segundos, sampler.cerrar_tramo()

**Líneas de código (`loc.py`).** Cuenta las líneas entre los marcadores de cada operación, con un criterio
explícito y reproducible.

In [ ]:
%%writefile /content/u2lib/src/emergencias123/loc.py
"""Conteo de líneas de código por operación del benchmark.

Criterio: se cuentan las líneas físicas entre `# >>> op:<nombre>` y `# <<< op:<nombre>` que no
estén vacías ni sean solo comentario. Una sentencia partida en varias líneas cuenta cada línea.
Las firmas `def` y los marcadores no se cuentan (quedan fuera del bloque).
"""
import re
from pathlib import Path

INICIO = re.compile(r'#\s*>>>\s*op:(\w+)')
FIN = re.compile(r'#\s*<<<\s*op:(\w+)')


def contar_loc(ruta_modulo):
    conteos, actual = {}, None
    for linea in Path(ruta_modulo).read_text(encoding='utf-8').splitlines():
        texto = linea.strip()
        if m := INICIO.search(texto):
            actual = m.group(1)
            conteos.setdefault(actual, 0)
            continue
        if FIN.search(texto):
            actual = None
            continue
        if actual and texto and not texto.startswith('#'):
            conteos[actual] += 1
    return conteos


def loc_por_herramienta():
    base = Path(__file__).resolve().parent
    return {h: contar_loc(base / f'ops_{h}.py') for h in ('pandas', 'polars', 'spark')}

**Operaciones en Pandas (`ops_pandas.py`).** Ejecución inmediata: cada llamada ya materializa.

In [ ]:
%%writefile /content/u2lib/src/emergencias123/ops_pandas.py
"""Operaciones del benchmark en Pandas (ejecución inmediata: cada llamada ya materializa)."""
import pandas as pd

from .normalizacion import CON_TILDE, SIN_TILDE

TILDES = str.maketrans(CON_TILDE, SIN_TILDE)


def _normalizar(serie):
    return serie.str.translate(TILDES).str.replace(r'\s+', ' ', regex=True).str.strip().str.upper()


def carga(rutas, separador=';', codificacion='utf-8-sig'):
    # >>> op:carga
    partes = [pd.read_csv(r, sep=separador, encoding=codificacion, dtype=str,
                          keep_default_na=False, na_values=['']) for r in rutas]
    df = pd.concat(partes, ignore_index=True)
    # <<< op:carga
    return df


def filtrado(df, localidades):
    # >>> op:filtrado
    df = df.assign(localidad_norm=_normalizar(df['LOCALIDAD']),
                   codigo_tipo=_normalizar(df['TIPO_INCIDENTE']).str.split(' - ', n=1).str[0])
    df = df[df['localidad_norm'].isin(localidades)].reset_index(drop=True)
    # <<< op:filtrado
    return df


def agregacion(df):
    # >>> op:agregacion
    conteo = df.groupby('codigo_tipo').size().reset_index(name='conteo')
    conteo = conteo.sort_values(['conteo', 'codigo_tipo'], ascending=[False, True], ignore_index=True)
    # <<< op:agregacion
    return conteo


def join(df, ruta_catalogo):
    # >>> op:join
    catalogo = pd.read_csv(ruta_catalogo, dtype=str, usecols=['codigo_tipo', 'descripcion_tipo', 'codigo_explicito'])
    catalogo['codigo_explicito'] = catalogo['codigo_explicito'] == 'True'
    enriquecido = df.merge(catalogo, on='codigo_tipo', how='left', validate='many_to_one')
    # <<< op:join
    return enriquecido


def exportacion(df, ruta_salida):
    # >>> op:exportacion
    df.to_parquet(ruta_salida / 'parte-0.parquet', index=False)
    # <<< op:exportacion


def filas(df):
    return len(df)


def a_dict_conteo(conteo):
    return dict(zip(conteo['codigo_tipo'], conteo['conteo'].astype(int)))


def sin_coincidencia(df):
    return int(df['descripcion_tipo'].isna().sum())

**Operaciones en Polars (`ops_polars.py`).** Modo eager por operación y modo lazy para el extremo a extremo.

In [ ]:
%%writefile /content/u2lib/src/emergencias123/ops_polars.py
"""Operaciones del benchmark en Polars.

Las mismas funciones sirven para DataFrame (modo por_operacion, eager) y LazyFrame
(modo extremo_a_extremo, perezoso): la API de expresiones es común a ambos.
"""
import polars as pl

from .normalizacion import CON_TILDE, SIN_TILDE


def _normalizar(columna):
    return (pl.col(columna).str.replace_many(list(CON_TILDE), list(SIN_TILDE))
            .str.replace_all(r'\s+', ' ').str.strip_chars().str.to_uppercase())


def carga(rutas, separador=';', perezoso=False):
    # >>> op:carga
    leer = pl.scan_csv if perezoso else pl.read_csv
    df = pl.concat([leer(r, separator=separador, infer_schema=False) for r in rutas])
    # <<< op:carga
    return df


def filtrado(df, localidades):
    # >>> op:filtrado
    df = df.with_columns(localidad_norm=_normalizar('LOCALIDAD'),
                         codigo_tipo=_normalizar('TIPO_INCIDENTE').str.split(' - ').list.first())
    df = df.filter(pl.col('localidad_norm').is_in(localidades))
    # <<< op:filtrado
    return df


def agregacion(df):
    # >>> op:agregacion
    conteo = df.group_by('codigo_tipo').agg(conteo=pl.len())
    conteo = conteo.sort(['conteo', 'codigo_tipo'], descending=[True, False])
    # <<< op:agregacion
    return conteo


def join(df, ruta_catalogo):
    # >>> op:join
    catalogo = pl.read_csv(ruta_catalogo, infer_schema=False,
                           columns=['codigo_tipo', 'descripcion_tipo', 'codigo_explicito'])
    catalogo = catalogo.with_columns(codigo_explicito=pl.col('codigo_explicito') == 'True')
    enriquecido = df.join(catalogo.lazy() if isinstance(df, pl.LazyFrame) else catalogo,
                          on='codigo_tipo', how='left', validate='m:1')
    # <<< op:join
    return enriquecido


def exportacion(df, ruta_salida):
    # >>> op:exportacion
    df.write_parquet(ruta_salida / 'parte-0.parquet')
    # <<< op:exportacion


def filas(df):
    return df.height


def a_dict_conteo(conteo):
    return dict(zip(conteo['codigo_tipo'].to_list(), conteo['conteo'].to_list()))


def sin_coincidencia(df):
    return int(df['descripcion_tipo'].null_count())

**Sesión de Spark (`spark_session.py`).** Configuración ajustada a los recursos que detecta: `local[*]` según
las vCPU, memoria del driver acotada a la RAM disponible, particiones de shuffle = 2 × vCPU, AQE y Arrow
activados, zona horaria de Bogotá y la interfaz web desactivada. Es Spark local en un único nodo, no un clúster.

In [ ]:
%%writefile /content/u2lib/src/emergencias123/spark_session.py
"""SparkSession configurada según los recursos reales del equipo donde corre.

En Colab, Spark corre en modo local (driver y ejecutores en una sola JVM de un único nodo):
no es un clúster. La configuración base es la de la guía, ajustada a vCPU y RAM detectados.
"""
import os

import psutil


def configuracion_spark(cfg):
    cpus = os.cpu_count() or 1
    ram_gb = psutil.virtual_memory().total / 2 ** 30
    memoria_driver = max(1, min(int(cfg.spark['driver_memory_max_gb']), int(ram_gb * 0.5)))
    return {
        'spark.master': f'local[{cpus}]',
        'spark.app.name': cfg.spark['app_name'],
        'spark.driver.memory': f'{memoria_driver}g',
        'spark.driver.maxResultSize': cfg.spark['driver_max_result_size'],
        'spark.sql.adaptive.enabled': 'true',
        'spark.sql.adaptive.coalescePartitions.enabled': 'true',
        'spark.sql.adaptive.skewJoin.enabled': 'true',
        'spark.sql.adaptive.advisoryPartitionSizeInBytes': cfg.spark['advisory_partition_size'],
        'spark.sql.execution.arrow.pyspark.enabled': 'true',
        'spark.sql.shuffle.partitions': str(cfg.spark['shuffle_partitions_por_cpu'] * cpus),
        'spark.sql.session.timeZone': cfg.spark['zona_horaria'],
        'spark.ui.enabled': 'false',
        'spark.ui.showConsoleProgress': 'false',
    }


def crear_sesion(cfg):
    """Crea la sesión. Debe llamarse en un proceso sin JVM previa para que driver.memory surta efecto."""
    from pyspark.sql import SparkSession

    conf = configuracion_spark(cfg)
    builder = SparkSession.builder
    for clave, valor in conf.items():
        builder = builder.config(clave, valor)
    spark = builder.getOrCreate()
    spark.sparkContext.setLogLevel('ERROR')
    return spark


def detener_sesion(spark):
    if spark is not None:
        spark.catalog.clearCache()
        spark.stop()

**Operaciones en PySpark (`ops_spark.py`).** Evaluación perezosa: nada se ejecuta hasta una acción. Por eso
el modo por operación materializa con `cache()+count()` y libera la caché al terminar.

In [ ]:
%%writefile /content/u2lib/src/emergencias123/ops_spark.py
"""Operaciones del benchmark en PySpark (evaluación perezosa: nada se ejecuta hasta una acción).

La materialización la decide el orquestador (benchmark.py): en el modo por_operacion se aplica
cache()+count() en cada frontera; en el extremo_a_extremo solo actúan las acciones finales.
"""
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, StructField, StructType

from .normalizacion import CON_TILDE, SIN_TILDE


def _normalizar(columna):
    return F.upper(F.trim(F.regexp_replace(F.translate(F.col(columna), CON_TILDE, SIN_TILDE), r'\s+', ' ')))


def esquema_texto(columnas):
    return StructType([StructField(c, StringType(), True) for c in columnas])


def carga(spark, rutas, columnas, separador=';'):
    # >>> op:carga
    df = spark.read.csv([str(r) for r in rutas], schema=esquema_texto(columnas), sep=separador,
                        header=True, encoding='UTF-8')
    # <<< op:carga
    return df


def filtrado(df, localidades):
    # >>> op:filtrado
    df = df.withColumns({'localidad_norm': _normalizar('LOCALIDAD'),
                         'codigo_tipo': F.split(_normalizar('TIPO_INCIDENTE'), ' - ', 2)[0]})
    df = df.filter(F.col('localidad_norm').isin(localidades))
    # <<< op:filtrado
    return df


def agregacion(df):
    # >>> op:agregacion
    conteo = df.groupBy('codigo_tipo').agg(F.count(F.lit(1)).alias('conteo'))
    conteo = conteo.orderBy(F.desc('conteo'), F.asc('codigo_tipo'))
    # <<< op:agregacion
    return conteo


def join(df, ruta_catalogo, spark):
    # >>> op:join
    catalogo = spark.read.csv(str(ruta_catalogo), header=True, encoding='UTF-8')
    catalogo = catalogo.select('codigo_tipo', 'descripcion_tipo',
                               (F.col('codigo_explicito') == 'True').alias('codigo_explicito'))
    enriquecido = df.join(F.broadcast(catalogo), on='codigo_tipo', how='left')
    # <<< op:join
    return enriquecido


def exportacion(df, ruta_salida):
    # >>> op:exportacion
    df.write.mode('overwrite').parquet(str(ruta_salida))
    # <<< op:exportacion


def filas(df):
    return df.count()


def a_dict_conteo(conteo):
    return {r['codigo_tipo']: int(r['conteo']) for r in conteo.collect()}


def sin_coincidencia(df):
    return df.filter(F.col('descripcion_tipo').isNull()).count()


def claves_catalogo_unicas(spark, ruta_catalogo):
    catalogo = spark.read.csv(str(ruta_catalogo), header=True)
    return catalogo.count() == catalogo.select('codigo_tipo').distinct().count()

**`measure_performance()` (`benchmark.py`).** Lanza cada repetición en un subproceso, mide cada operación y el
total, y agrega una línea JSON por medición a `benchmark.jsonl` (herramienta, modo, operación, segundos, memoria
base y pico, filas de entrada y salida, LOC, caché y errores).

In [ ]:
%%writefile /content/u2lib/src/emergencias123/benchmark.py
"""Benchmark Pandas vs Polars vs PySpark con measure_performance().

Cada (herramienta, modo, ejecución) corre en un subproceso Python NUEVO (`python -m
emergencias123.benchmark ...`): no se reutilizan DataFrames, cachés ni JVM entre repeticiones.
El orquestador (`ejecutar_benchmark`) hace 1 calentamiento descartado por herramienta y modo,
y luego N repeticiones con el orden de herramientas rotado en cada ronda.

Modos:
- por_operacion: cada operación se materializa y se mide por separado. En Spark se aplica
  cache()+count() en cada frontera para que la operación siguiente no recompute las anteriores.
- extremo_a_extremo: el pipeline completo con la mínima materialización (Polars perezoso con
  collect_all; Spark con un DAG único y solo las acciones finales, con caché del filtrado porque
  lo consumen dos acciones: la agregación y la exportación).
- extremo_a_extremo_caliente (complementario): igual que el anterior, pero se mide la SEGUNDA
  pasada dentro del mismo proceso, tras limpiar la caché de datos. Separa el coste de arranque en
  frío (JVM recién creada) del coste en régimen estable. No sustituye a los modos principales.
"""
import argparse
import json
import logging
import os
import shutil
import subprocess
import sys
import time
from datetime import datetime
from pathlib import Path

from .config import cargar_config
from .loc import loc_por_herramienta
from .memoria import MemorySampler, medir

OPERACIONES = ['carga', 'filtrado', 'agregacion', 'join', 'exportacion']
log = logging.getLogger(__name__)


def _filas_fisicas(cfg):
    """Registros de datos de los CSV (líneas no vacías menos el encabezado), para los modos sin conteo intermedio."""
    total = 0
    for ruta in cfg.rutas_csv:
        with open(ruta, 'rb') as f:
            total += sum(1 for linea in f if linea.strip()) - 1
    return total


class _Registro:
    def __init__(self, base):
        self.base, self.filas = base, []

    def agregar(self, operacion, segundos, memoria, entrada, salida, accion, cache=False, **extra):
        self.filas.append({**self.base, 'operacion': operacion, 'segundos': round(segundos, 6), **memoria,
                           'filas_entrada': entrada, 'filas_salida': salida, 'estado': 'ok', 'error': None,
                           'accion_materializacion': accion, 'usa_cache': cache, **extra})


# ---------- Pandas y Polars: modo por_operacion (eager) ----------
def _por_operacion_eager(ops, s, cfg, ruta, reg, **kw_carga):
    locs, cat = cfg.benchmark['localidades_filtro'], cfg.ruta_catalogo_tipos
    df, t, m = medir(s, ops.carga, cfg.rutas_csv, cfg.datos['separador'], **kw_carga)
    reg.agregar('carga', t, m, None, ops.filas(df), 'eager')
    dff, t, m = medir(s, ops.filtrado, df, locs)
    reg.agregar('filtrado', t, m, ops.filas(df), ops.filas(dff), 'eager')
    conteo, t, m = medir(s, ops.agregacion, dff)
    reg.agregar('agregacion', t, m, ops.filas(dff), ops.filas(conteo), 'eager',
                resultado_agregacion=ops.a_dict_conteo(conteo))
    enr, t, m = medir(s, ops.join, dff, cat)
    reg.agregar('join', t, m, ops.filas(dff), ops.filas(enr), 'eager', sin_coincidencia=ops.sin_coincidencia(enr))
    _, t, m = medir(s, ops.exportacion, enr, ruta)
    reg.agregar('exportacion', t, m, ops.filas(enr), ops.filas(enr), 'escritura parquet')


def _pandas_por_operacion(s, cfg, ruta, reg):
    from . import ops_pandas
    _por_operacion_eager(ops_pandas, s, cfg, ruta, reg, codificacion=cfg.datos['codificacion'])


def _polars_por_operacion(s, cfg, ruta, reg):
    from . import ops_polars
    _por_operacion_eager(ops_polars, s, cfg, ruta, reg)


# ---------- Pandas y Polars: modo extremo_a_extremo ----------
def _pandas_extremo(s, cfg, ruta, reg):
    from . import ops_pandas as ops

    def pipeline():
        dff = ops.filtrado(ops.carga(cfg.rutas_csv, cfg.datos['separador'], cfg.datos['codificacion']),
                           cfg.benchmark['localidades_filtro'])
        conteo = ops.agregacion(dff)
        enr = ops.join(dff, cfg.ruta_catalogo_tipos)
        ops.exportacion(enr, ruta)
        return ops.a_dict_conteo(conteo), ops.filas(enr), ops.sin_coincidencia(enr)

    (conteo, n, sin), t, m = medir(s, pipeline)
    reg.agregar('extremo_a_extremo', t, m, _filas_fisicas(cfg), n, 'eager + escritura parquet',
                resultado_agregacion=conteo, sin_coincidencia=sin)


def _polars_extremo(s, cfg, ruta, reg):
    import polars as pl

    from . import ops_polars as ops

    def pipeline():
        lff = ops.filtrado(ops.carga(cfg.rutas_csv, cfg.datos['separador'], perezoso=True),
                           cfg.benchmark['localidades_filtro'])
        conteo, enr = pl.collect_all([ops.agregacion(lff), ops.join(lff, cfg.ruta_catalogo_tipos)])
        ops.exportacion(enr, ruta)
        return ops.a_dict_conteo(conteo), ops.filas(enr), ops.sin_coincidencia(enr)

    (conteo, n, sin), t, m = medir(s, pipeline)
    reg.agregar('extremo_a_extremo', t, m, _filas_fisicas(cfg), n, 'collect_all (subplan común) + write_parquet',
                resultado_agregacion=conteo, sin_coincidencia=sin)


# ---------- Spark ----------
def _materializar(df):
    df = df.cache()
    return df, df.count()


def _spark_por_operacion(s, cfg, ruta, reg, spark):
    from . import ops_spark as ops
    cols, locs = cfg.esquema['columnas'], cfg.benchmark['localidades_filtro']
    (df, n0), t, m = medir(s, lambda: _materializar(ops.carga(spark, cfg.rutas_csv, cols, cfg.datos['separador'])))
    reg.agregar('carga', t, m, None, n0, 'cache()+count()', cache=True)
    (dff, n1), t, m = medir(s, lambda: _materializar(ops.filtrado(df, locs)))
    reg.agregar('filtrado', t, m, n0, n1, 'cache()+count()', cache=True)
    df.unpersist(blocking=True)
    conteo, t, m = medir(s, lambda: ops.a_dict_conteo(ops.agregacion(dff)))
    reg.agregar('agregacion', t, m, n1, len(conteo), 'collect()', cache=True, resultado_agregacion=conteo)
    (enr, n2), t, m = medir(s, lambda: _materializar(ops.join(dff, cfg.ruta_catalogo_tipos, spark)))
    reg.agregar('join', t, m, n1, n2, 'cache()+count()', cache=True, sin_coincidencia=ops.sin_coincidencia(enr))
    _, t, m = medir(s, ops.exportacion, enr, ruta)
    reg.agregar('exportacion', t, m, n2, n2, 'write.parquet()', cache=True)
    enr.unpersist(blocking=True)
    dff.unpersist(blocking=True)


def _spark_extremo(s, cfg, ruta, reg, spark):
    from . import ops_spark as ops
    cols, locs = cfg.esquema['columnas'], cfg.benchmark['localidades_filtro']
    estado = {}

    def pipeline():
        dff = ops.filtrado(ops.carga(spark, cfg.rutas_csv, cols, cfg.datos['separador']), locs).cache()
        conteo = ops.a_dict_conteo(ops.agregacion(dff))
        enr = ops.join(dff, cfg.ruta_catalogo_tipos, spark)
        ops.exportacion(enr, ruta)
        estado.update(dff=dff, enr=enr)
        return conteo

    conteo, t, m = medir(s, pipeline)
    n = estado['enr'].count()
    reg.agregar('extremo_a_extremo', t, m, _filas_fisicas(cfg), n,
                'collect() de la agregación + write.parquet(); filtrado en caché (lo usan ambas acciones)',
                cache=True, resultado_agregacion=conteo, sin_coincidencia=ops.sin_coincidencia(estado['enr']))
    estado['dff'].unpersist(blocking=True)


def _caliente(ejecutor):
    """Modo complementario: una pasada previa sin medir en el MISMO proceso (JIT, clases y codegen de la
    JVM ya cargados; imports y pools de hilos ya creados), limpieza de la caché de datos y luego la
    pasada medida. Aísla el coste de arranque en frío del coste en régimen estable."""
    def ejecutar(s, cfg, ruta, reg, *spark):
        previa = ruta.with_name(ruta.name + '_previa')
        shutil.rmtree(previa, ignore_errors=True)
        if not spark:
            previa.mkdir(parents=True, exist_ok=True)
        ejecutor(s, cfg, previa, _Registro({}), *spark)
        if spark:
            spark[0].catalog.clearCache()
        ejecutor(s, cfg, ruta, reg, *spark)
    return ejecutar


EJECUTORES = {
    ('pandas', 'por_operacion'): _pandas_por_operacion,
    ('pandas', 'extremo_a_extremo'): _pandas_extremo,
    ('pandas', 'extremo_a_extremo_caliente'): _caliente(_pandas_extremo),
    ('polars', 'por_operacion'): _polars_por_operacion,
    ('polars', 'extremo_a_extremo'): _polars_extremo,
    ('polars', 'extremo_a_extremo_caliente'): _caliente(_polars_extremo),
    ('spark', 'por_operacion'): _spark_por_operacion,
    ('spark', 'extremo_a_extremo'): _spark_extremo,
    ('spark', 'extremo_a_extremo_caliente'): _caliente(_spark_extremo),
}


def _registro_total(filas, loc_total, operacion='total', extra_segundos=0.0):
    medidas = [f for f in filas if f['operacion'] in OPERACIONES + ['extremo_a_extremo']]
    if not medidas:
        return None
    total = dict(medidas[0])
    total.update(operacion=operacion, segundos=round(sum(f['segundos'] for f in medidas) + extra_segundos, 6),
                 mem_sis_pico_pct=max(f['mem_sis_pico_pct'] for f in medidas),
                 rss_pico_mb=max(f['rss_pico_mb'] for f in medidas),
                 n_muestras=sum(f['n_muestras'] for f in medidas),
                 filas_salida=medidas[-1]['filas_salida'], loc=loc_total,
                 accion_materializacion='suma de las operaciones medidas', resultado_agregacion=None)
    return total


def measure_performance(herramienta, ejecucion, modo, calentamiento, cfg):
    """Ejecuta y mide una repetición completa del benchmark en el proceso actual.

    Devuelve una lista de registros: uno por operación (o uno de extremo a extremo), más `total`
    (de la carga a la exportación) y, en Spark, `inicio_sesion` y `total_con_arranque`.
    """
    loc = loc_por_herramienta()[herramienta]
    loc_total = sum(loc[o] for o in OPERACIONES)
    ruta = cfg.dir_resultados / 'parquet' / 'benchmark' / herramienta / modo / f'e{ejecucion}{"_cal" if calentamiento else ""}'
    shutil.rmtree(ruta, ignore_errors=True)
    if herramienta != 'spark':
        ruta.mkdir(parents=True, exist_ok=True)
    base = {'herramienta': herramienta, 'modo': modo, 'ejecucion': ejecucion, 'calentamiento': calentamiento,
            'entorno': cfg.entorno, 'ts_inicio': datetime.now().isoformat(timespec='seconds'),
            'ruta_salida': str(ruta)}
    reg = _Registro(base)
    spark, arranque = None, 0.0
    with MemorySampler(cfg.benchmark['intervalo_muestreo_s']) as s:
        try:
            if herramienta == 'spark':
                from .spark_session import crear_sesion
                spark, arranque, m = medir(s, crear_sesion, cfg)
                reg.agregar('inicio_sesion', arranque, m, None, None, 'creación de SparkSession (JVM)')
                EJECUTORES[(herramienta, modo)](s, cfg, ruta, reg, spark)
            else:
                EJECUTORES[(herramienta, modo)](s, cfg, ruta, reg)
        except Exception as error:  # se registra como medición fallida, no se oculta
            log.exception('Fallo en %s/%s/%s', herramienta, modo, ejecucion)
            reg.filas.append({**base, 'operacion': 'error', 'segundos': None, 'estado': 'error',
                              'error': f'{type(error).__name__}: {error}'})
        finally:
            if spark is not None:
                from .spark_session import detener_sesion
                detener_sesion(spark)
    for fila in reg.filas:
        fila.setdefault('loc', loc.get(fila['operacion'], loc_total if fila['operacion'] == 'extremo_a_extremo' else None))
    if not any(f['estado'] == 'error' for f in reg.filas):
        reg.filas.append(_registro_total(reg.filas, loc_total))
        if herramienta == 'spark':
            reg.filas.append(_registro_total(reg.filas[:-1], loc_total, 'total_con_arranque', arranque))
    return reg.filas


def guardar_jsonl(registros, ruta):
    ruta.parent.mkdir(parents=True, exist_ok=True)
    with open(ruta, 'a', encoding='utf-8') as f:
        for r in registros:
            f.write(json.dumps(r, ensure_ascii=False) + '\n')


def ejecutar_subproceso(herramienta, modo, ejecucion, calentamiento, salida, timeout=1800):
    src = str(Path(__file__).resolve().parents[1])
    env = dict(os.environ, PYTHONPATH=src + os.pathsep + os.environ.get('PYTHONPATH', ''))
    cmd = [sys.executable, '-m', 'emergencias123.benchmark', '--herramienta', herramienta, '--modo', modo,
           '--ejecucion', str(ejecucion), '--salida', str(salida)] + (['--calentamiento'] if calentamiento else [])
    t0 = time.perf_counter()
    r = subprocess.run(cmd, env=env, capture_output=True, text=True, timeout=timeout)
    return {'herramienta': herramienta, 'modo': modo, 'ejecucion': ejecucion, 'calentamiento': calentamiento,
            'codigo_salida': r.returncode, 'segundos_proceso': round(time.perf_counter() - t0, 2),
            'stderr_final': r.stderr[-1500:] if r.returncode else ''}


def plan_benchmark(cfg, herramientas=None, modos=None, repeticiones=None):
    """Orden de ejecución: calentamientos y luego rondas con rotación del orden de herramientas."""
    herramientas = herramientas or cfg.benchmark['herramientas']
    modos = modos or cfg.benchmark['modos']
    repeticiones = repeticiones or cfg.benchmark['repeticiones']
    plan = []
    for modo in modos:
        for h in herramientas:
            plan += [(h, modo, c + 1, True) for c in range(cfg.benchmark['calentamientos'])]
        for r in range(repeticiones):
            k = r % len(herramientas)
            plan += [(h, modo, r + 1, False) for h in herramientas[k:] + herramientas[:k]]
    return plan


def ejecutar_benchmark(cfg, salida, herramientas=None, modos=None, repeticiones=None, al_avanzar=None):
    resultados = []
    plan = plan_benchmark(cfg, herramientas, modos, repeticiones)
    for i, paso in enumerate(plan, 1):
        res = ejecutar_subproceso(*paso, salida=salida)
        resultados.append(res)
        if al_avanzar:
            al_avanzar(i, len(plan), res)
    return resultados


def main():
    p = argparse.ArgumentParser(description='Una repetición medida del benchmark (en este proceso).')
    p.add_argument('--herramienta', required=True, choices=['pandas', 'polars', 'spark'])
    p.add_argument('--modo', required=True, choices=['por_operacion', 'extremo_a_extremo', 'extremo_a_extremo_caliente'])
    p.add_argument('--ejecucion', type=int, required=True)
    p.add_argument('--calentamiento', action='store_true')
    p.add_argument('--salida', required=True, type=Path)
    a = p.parse_args()
    logging.basicConfig(level=logging.WARNING)
    cfg = cargar_config()
    registros = measure_performance(a.herramienta, a.ejecucion, a.modo, a.calentamiento, cfg)
    guardar_jsonl(registros, a.salida)
    sys.exit(1 if any(r['estado'] == 'error' for r in registros) else 0)


if __name__ == '__main__':
    main()

**Equivalencia (`equivalencia.py`).** Relee los Parquet de cada herramienta y compara filas, esquema lógico,
conteo por tipo y la huella SHA-256 del contenido canónico, fuera del tiempo medido.

In [ ]:
%%writefile /content/u2lib/src/emergencias123/equivalencia.py
"""Comprobación de equivalencia lógica entre las salidas Parquet de las tres herramientas.

Se ejecuta fuera de la medición. Compara: número de filas, conjunto de columnas, tipos lógicos,
agregación por codigo_tipo y SHA-256 del contenido canónico (valores como texto, columnas en
orden fijo, filas ordenadas). Los Parquet de Spark (directorio de part-files) y de Pandas/Polars
(un archivo) se leen igual, con pyarrow.
"""
import hashlib
import json

import pyarrow.dataset as ds
import pyarrow.types as pat


def tipo_logico(tipo):
    if pat.is_string(tipo) or pat.is_large_string(tipo) or pat.is_string_view(tipo):
        return 'texto'
    if pat.is_boolean(tipo):
        return 'booleano'
    if pat.is_integer(tipo):
        return 'entero'
    if pat.is_floating(tipo):
        return 'decimal'
    if pat.is_timestamp(tipo):
        return 'fecha_hora'
    return str(tipo)


def leer_tabla(ruta):
    archivos = [p for p in ruta.rglob('*.parquet')] if ruta.is_dir() else [ruta]
    return ds.dataset([str(p) for p in archivos], format='parquet').to_table()


def huella(tabla):
    columnas = sorted(tabla.column_names)
    datos = tabla.select(columnas).to_pydict()
    filas = sorted('\x1f'.join('' if datos[c][i] is None else str(datos[c][i]) for c in columnas)
                   for i in range(tabla.num_rows))
    h = hashlib.sha256()
    for f in filas:
        h.update(f.encode('utf-8') + b'\n')
    return h.hexdigest()


def resumen_salida(ruta):
    tabla = leer_tabla(ruta)
    conteo = {}
    for codigo in tabla.column('codigo_tipo').to_pylist():
        conteo[codigo] = conteo.get(codigo, 0) + 1
    return {
        'filas': tabla.num_rows,
        'esquema': {c: tipo_logico(tabla.schema.field(c).type) for c in sorted(tabla.column_names)},
        'agregacion': dict(sorted(conteo.items())),
        'sha256': huella(tabla),
    }


def comparar(rutas_por_herramienta):
    """{'pandas': Path, ...} -> dict con resúmenes y veredicto por criterio."""
    resumenes = {h: resumen_salida(r) for h, r in rutas_por_herramienta.items()}
    criterios = {}
    for clave in ('filas', 'esquema', 'agregacion', 'sha256'):
        valores = {json.dumps(r[clave], sort_keys=True) for r in resumenes.values()}
        criterios[clave] = len(valores) == 1
    return {'resumenes': resumenes, 'criterios': criterios, 'equivalentes': all(criterios.values())}

**Gráficos (`reportes/graficos.py`).** Barras con desviación estándar a 300 dpi; el notebook de informes usa el
mismo módulo para las figuras de los documentos.

In [ ]:
%%writefile /content/u2lib/src/emergencias123/reportes/graficos.py
"""Gráficos del benchmark y del PoC (matplotlib, 300 dpi).

Paleta categórica validada (orden fijo, color ligado a la herramienta y no a su posición):
Pandas azul, Polars naranja, Spark aqua. El aqua queda bajo 3:1 de contraste sobre la superficie,
por eso cada barra lleva su valor rotulado. Nunca se usan dos ejes Y: cada magnitud va en su panel.
"""
import matplotlib

matplotlib.use('Agg')
import matplotlib.pyplot as plt  # noqa: E402
import pandas as pd  # noqa: E402

COLORES = {'pandas': '#2a78d6', 'polars': '#eb6834', 'spark': '#1baf7a'}
NOMBRES = {'pandas': 'Pandas', 'polars': 'Polars', 'spark': 'PySpark'}
ORDEN = ['pandas', 'polars', 'spark']
SUPERFICIE, TINTA, TINTA_2, MUTED, GRID = '#fcfcfb', '#0b0b0b', '#52514e', '#898781', '#e1e0d9'
OPERACIONES = ['carga', 'filtrado', 'agregacion', 'join', 'exportacion']
ETIQUETAS_OP = {'carga': 'Carga', 'filtrado': 'Filtrado', 'agregacion': 'Agregación', 'join': 'Join',
                'exportacion': 'Exportación'}


def _estilo(ax, titulo, eje):
    ax.set_facecolor(SUPERFICIE)
    ax.set_title(titulo, loc='left', fontsize=11, color=TINTA, fontweight='bold', pad=10)
    ax.set_xlabel(eje, color=TINTA_2, fontsize=9)
    ax.tick_params(colors=TINTA_2, labelsize=9, length=0)
    for lado in ('top', 'right', 'left'):
        ax.spines[lado].set_visible(False)
    ax.spines['bottom'].set_color(GRID)
    ax.grid(axis='x', color=GRID, linewidth=0.8)
    ax.set_axisbelow(True)


def _rotular(ax, barras, valores, desv, fmt):
    for b, v, e in zip(barras, valores, desv):
        ax.annotate(fmt.format(v), (b.get_width() + (e or 0), b.get_y() + b.get_height() / 2), xytext=(6, 0),
                    textcoords='offset points', va='center', fontsize=8, color=TINTA)


def _barras_h(ax, etiquetas, medias, desv, colores, fmt):
    pos = range(len(etiquetas))
    barras = ax.barh(list(pos), medias, xerr=desv, color=colores, height=0.6,
                     error_kw={'ecolor': MUTED, 'elinewidth': 1, 'capsize': 3})
    ax.set_yticks(list(pos), etiquetas)
    ax.invert_yaxis()
    _rotular(ax, barras, medias, desv, fmt)
    limite = max(m + (d or 0) for m, d in zip(medias, desv)) * 1.45
    ax.set_xlim(0, min(limite, 100) if '%' in fmt else limite)


def _figura(ncols, ancho, alto, nrows=1):
    fig, axes = plt.subplots(nrows, ncols, figsize=(ancho, alto), facecolor=SUPERFICIE, squeeze=False)
    return fig, list(axes.flat)


def _guardar(fig, ruta, titulo, nota):
    fig.suptitle(titulo, x=0.01, ha='left', fontsize=13, fontweight='bold', color=TINTA)
    fig.text(0.01, 0.005, nota, fontsize=7.5, color=TINTA_2, ha='left', va='bottom', wrap=True)
    fig.tight_layout(rect=(0, 0.06, 1, 0.94))
    fig.savefig(ruta, dpi=300, facecolor=SUPERFICIE)
    plt.close(fig)
    return ruta


def grafico_operaciones(medidas, ruta):
    """Tiempo medio por operación (modo por_operacion), un panel por operación (escalas propias)."""
    d = medidas[(medidas['modo'] == 'por_operacion') & medidas['operacion'].isin(OPERACIONES)]
    est = d.groupby(['operacion', 'herramienta'])['segundos'].agg(['mean', 'std']).reset_index()
    fig, axes = _figura(len(OPERACIONES), 17, 3.4)
    for ax, op in zip(axes, OPERACIONES):
        e = est[est['operacion'] == op].set_index('herramienta').reindex(ORDEN).dropna(subset=['mean'])
        _barras_h(ax, [NOMBRES[h] for h in e.index], e['mean'].tolist(), e['std'].fillna(0).tolist(),
                  [COLORES[h] for h in e.index], '{:.3f} s')
        _estilo(ax, ETIQUETAS_OP[op], 'segundos (media ± desv. est.)')
    n = int(d.groupby('herramienta')['ejecucion'].nunique().min())
    return _guardar(fig, ruta, 'Benchmark: tiempo por operación (modo por operación)',
                    f'{n} repeticiones medidas por herramienta, cada una en un proceso nuevo; calentamiento excluido. '
                    'Spark materializa cada operación con cache()+count(); el arranque de la sesión se mide aparte.')


def grafico_totales(medidas, ruta, por_fila=None):
    """Tiempo total por modo y memoria pico (RSS proceso+hijos y % del sistema), en paneles separados.

    `por_fila` reparte los paneles en varias filas (versión para documentos de página vertical).
    """
    tot = medidas[medidas['operacion'] == 'total']
    paneles = [
        ('Total extremo a extremo', tot[tot['modo'] == 'extremo_a_extremo'], 'segundos', 'segundos (media ± d.e.)', '{:.2f} s'),
        ('Total por operación (suma)', tot[tot['modo'] == 'por_operacion'], 'segundos', 'segundos (media ± d.e.)', '{:.2f} s'),
        ('Extremo a extremo, proceso caliente*', tot[tot['modo'] == 'extremo_a_extremo_caliente'], 'segundos',
         'segundos (media ± d.e.)', '{:.2f} s'),
        ('RSS pico proceso + hijos (JVM)', tot[tot['modo'] == 'extremo_a_extremo'], 'rss_pico_mb', 'MB (media ± d.e.)', '{:.0f} MB'),
        ('Memoria del sistema, pico', tot[tot['modo'] == 'extremo_a_extremo'], 'mem_sis_pico_pct', '% de la RAM total (media ± d.e.)', '{:.1f} %'),
    ]
    paneles = [p for p in paneles if len(p[1])]
    columnas = min(por_fila or len(paneles), len(paneles))
    filas = -(-len(paneles) // columnas)
    fig, axes = _figura(columnas, 4.3 * columnas, 3.2 * filas + 0.4, nrows=filas)
    for ax in axes[len(paneles):]:
        ax.set_visible(False)
    for ax, (titulo, d, col, eje, fmt) in zip(axes, paneles):
        e = d.groupby('herramienta')[col].agg(['mean', 'std']).reindex(ORDEN).dropna(subset=['mean'])
        _barras_h(ax, [NOMBRES[h] for h in e.index], e['mean'].tolist(), e['std'].fillna(0).tolist(),
                  [COLORES[h] for h in e.index], fmt)
        _estilo(ax, titulo, eje)
    return _guardar(fig, ruta, 'Benchmark: tiempo total y memoria (de la carga a la exportación)',
                    'Tiempos sin el arranque de SparkSession. *Proceso caliente: 2.ª pasada en el mismo proceso tras una '
                    'pasada sin medir (complementario). Memoria muestreada cada 20 ms; el % del sistema incluye todo el '
                    'runtime de Colab y no es memoria exclusiva de la herramienta.')


def grafico_poc(metricas, ruta):
    """Comparación PoC PySpark vs Pandas (mismo alcance): tiempo, memoria del sistema y RSS."""
    paneles = [
        ('Tiempo total carga→exportación', 'tiempo_total_s', 'segundos (media ± d.e.)', '{:.2f} s'),
        ('Memoria del sistema, pico', 'mem_sis_pico_pct', '% de la RAM total (media ± d.e.)', '{:.1f} %'),
        ('RSS pico proceso + hijos (JVM)', 'rss_pico_mb', 'MB (media ± d.e.)', '{:.0f} MB'),
    ]
    fig, axes = _figura(3, 14, 3.0)
    for ax, (titulo, col, eje, fmt) in zip(axes, paneles):
        e = metricas.groupby('herramienta')[col].agg(['mean', 'std']).reindex(['pandas', 'spark']).dropna(subset=['mean'])
        _barras_h(ax, [NOMBRES[h] for h in e.index], e['mean'].tolist(), e['std'].fillna(0).tolist(),
                  [COLORES[h] for h in e.index], fmt)
        _estilo(ax, titulo, eje)
    n = int(metricas.groupby('herramienta')['ejecucion'].nunique().min())
    return _guardar(fig, ruta, 'PoC: PySpark frente a Pandas con el mismo alcance (3 meses, Colab)',
                    f'{n} ejecuciones por herramienta, cada una en un proceso nuevo. Valores más bajos son mejores. '
                    'El tiempo de Spark excluye el arranque de la sesión, que se reporta aparte.')


def cargar_mediciones(ruta_jsonl):
    df = pd.read_json(ruta_jsonl, lines=True)
    return df[(df['estado'] == 'ok') & (~df['calentamiento'].astype(bool))] if 'calentamiento' in df else df

**Controles de calidad (`validaciones.py`).** Comunes a Spark y Pandas. Cada control tiene severidad
(bloqueante, advertencia o informativo); aquí están también las fórmulas de las tasas y el log `.txt`.

In [ ]:
%%writefile /content/u2lib/src/emergencias123/validaciones.py
"""Controles de calidad del PoC, comunes a Spark y Pandas, y fórmulas de las tasas.

Los pipelines producen un diccionario de `hechos` medidos y aquí se evalúan los controles.
- Tasa de éxito de validaciones (%) = controles aprobados / controles evaluados x 100, donde
  evaluados = bloqueantes + advertencias. Los informativos se reportan pero no entran en la tasa.
- Registros válidos (%) = aceptados / leídos x 100. Es otra métrica y no se mezcla con la anterior.

Regla de limpieza (decisión del usuario, 2026-11-10): solo se ELIMINAN las copias duplicadas exactas (100 % de los
atributos iguales). En los hechos, `aceptados` = registros conservados y `rechazados` = copias eliminadas. Los nulos
y demás anomalías se conservan y quedan marcados en `observacion_calidad` (ver MOTIVOS).
"""
import csv
from pathlib import Path

import pyarrow.dataset as ds

COLUMNAS_DETALLE = [
    'numero_incidente', 'fecha_inicio', 'fecha_recepcion', 'periodo', 'archivo_origen', 'en_mes_archivo',
    'codigo_localidad', 'localidad_norm', 'localidad', 'edad', 'unidad', 'edad_anios', 'genero',
    'paciente_completo', 'tipo_incidente', 'codigo_tipo', 'descripcion_tipo', 'codigo_explicito',
    'codigo_reconocido', 'prioridad', 'observacion_calidad', 'incidente_multidespacho',
]
MOTIVOS = ['registro_corrupto', 'numero_incidente_nulo', 'fecha_inicio_invalida', 'localidad_sin_dato',
           'localidad_fuera_catalogo', 'codigo_localidad_inconsistente', 'tipo_incidente_nulo', 'edad_fuera_rango']


def prevalidar(cfg):
    """Controles sobre los archivos antes de procesar: presencia, encabezados y filas físicas."""
    esperadas = cfg.esquema['columnas']
    presentes, faltantes, filas = 0, {}, {}
    for ruta in cfg.rutas_csv:
        if not ruta.exists():
            continue
        presentes += 1
        with open(ruta, encoding=cfg.datos['codificacion']) as f:
            encabezado = f.readline().strip().split(cfg.datos['separador'])
            filas[ruta.name] = sum(1 for linea in f if linea.strip())
        faltan = [c for c in esperadas if c not in encabezado]
        if faltan:
            faltantes[ruta.name] = faltan
    return {'archivos_esperados': len(cfg.rutas_csv), 'archivos_presentes': presentes,
            'columnas_faltantes': faltantes, 'filas_fisicas_por_archivo': filas,
            'filas_fisicas': sum(filas.values())}


def leer_localidades(cfg):
    with open(cfg.ruta_localidades, encoding='utf-8', newline='') as f:
        return {r['localidad_norm']: int(r['codigo_localidad']) for r in csv.DictReader(f)}


def _archivos_parquet(ruta):
    return sorted(p for p in Path(ruta).rglob('*.parquet'))


def verificar_salidas(salida):
    """Relee los Parquet exportados con pyarrow (lector independiente de la herramienta que los escribió)."""
    detalle = salida / 'detalle'
    archivos = _archivos_parquet(detalle)
    tabla = ds.dataset(str(detalle), format='parquet', partitioning='hive').to_table()
    rechazados = _archivos_parquet(salida / 'duplicados_eliminados')
    n_rech = ds.dataset([str(p) for p in rechazados], format='parquet').count_rows() if rechazados else 0
    return {
        'parquet_conteo': tabla.num_rows,
        'parquet_columnas': sorted(tabla.column_names),
        'parquet_archivos': len(archivos),
        'parquet_bytes': sum(p.stat().st_size for p in archivos),
        'parquet_particiones': sorted({p.parent.name for p in archivos}),
        'parquet_rechazados_conteo': n_rech,
        'parquet_suma_agregado_releido': sum(
            ds.dataset(str(salida / 'agregados' / 'periodo_localidad_tipo'), format='parquet')
            .to_table(columns=['conteo']).column('conteo').to_pylist()),
    }


def _control(id_, descripcion, severidad, valor, esperado, aprobado):
    return {'id': id_, 'descripcion': descripcion, 'severidad': severidad,
            'valor': valor, 'esperado': esperado, 'aprobado': bool(aprobado)}


def evaluar(h):
    """Lista de controles a partir de los hechos medidos (pipeline + verificación de salidas)."""
    c = [
        _control('V01', 'Archivos de entrada presentes', 'bloqueante',
                 h['archivos_presentes'], h['archivos_esperados'], h['archivos_presentes'] == h['archivos_esperados']),
        _control('V02', 'Columnas requeridas presentes en cada archivo', 'bloqueante',
                 h['columnas_faltantes'] or 'ninguna faltante', 'ninguna faltante', not h['columnas_faltantes']),
        _control('V03', 'Registros leídos = filas físicas de datos de los CSV', 'bloqueante',
                 h['leidos'], h['filas_fisicas'], h['leidos'] == h['filas_fisicas']),
        _control('V04', 'Leídos = conservados + duplicados exactos eliminados', 'bloqueante',
                 h['aceptados'] + h['rechazados'], h['leidos'], h['aceptados'] + h['rechazados'] == h['leidos']),
        _control('V05', 'Fechas de inicio no parseables', 'advertencia',
                 h['fechas_invalidas'], 0, h['fechas_invalidas'] == 0),
        _control('V06', 'Conservados fuera de la ventana [mes del archivo, fin de mes + tolerancia]', 'advertencia',
                 h['fuera_ventana_aceptados'], 0, h['fuera_ventana_aceptados'] == 0),
        _control('V07', 'Nulos en campos críticos sin observación de calidad (nulo silencioso)', 'bloqueante',
                 h['nulos_criticos_sin_observacion'], 0, h['nulos_criticos_sin_observacion'] == 0),
        _control('V08', 'Códigos de tipo no reconocidos por el catálogo derivado', 'advertencia',
                 h['codigos_no_reconocidos'], 0, h['codigos_no_reconocidos'] == 0),
        _control('V09', 'Clave del catálogo única y join sin multiplicar filas', 'bloqueante',
                 f"clave_unica={h['catalogo_clave_unica']}; {h['filas_antes_join']}->{h['filas_despues_join']}",
                 'clave_unica=True; filas iguales',
                 h['catalogo_clave_unica'] and h['filas_antes_join'] == h['filas_despues_join']),
        _control('V10', 'Inconsistencias código↔nombre de localidad', 'advertencia',
                 h['inconsistencias_localidad'], 0, h['inconsistencias_localidad'] == 0),
        _control('V11', 'Edades fuera de rango (conservadas y marcadas)', 'advertencia',
                 h['edades_fuera_rango_aceptados'], 0, h['edades_fuera_rango_aceptados'] == 0),
        _control('V12', 'Suma del agregado periodo×localidad×tipo = conservados', 'bloqueante',
                 h['suma_agregado_principal'], h['aceptados'], h['suma_agregado_principal'] == h['aceptados']),
        _control('V13', 'Parquet releído: filas = conservados', 'bloqueante',
                 h['parquet_conteo'], h['aceptados'], h['parquet_conteo'] == h['aceptados']),
        _control('V14', 'Parquet releído: columnas esperadas', 'bloqueante',
                 len(h['parquet_columnas']), len(COLUMNAS_DETALLE),
                 sorted(h['parquet_columnas']) == sorted(COLUMNAS_DETALLE)),
        _control('V15', 'Parquet de duplicados eliminados releído = duplicados eliminados', 'bloqueante',
                 h['parquet_rechazados_conteo'], h['rechazados'], h['parquet_rechazados_conteo'] == h['rechazados']),
        _control('V16', 'Agregado releído desde Parquet = conservados', 'bloqueante',
                 h['parquet_suma_agregado_releido'], h['aceptados'], h['parquet_suma_agregado_releido'] == h['aceptados']),
        _control('V17', 'Copias duplicadas exactas eliminadas (100 % de atributos iguales)', 'informativo',
                 h['duplicados_exactos'], 'se reporta', True),
        _control('V18', 'Filas de incidentes con varios despachos', 'informativo',
                 h['filas_incidente_repetido'], 'se reporta', True),
        _control('V19', 'Registros conservados con observación de calidad, por motivo', 'informativo',
                 h['observaciones_por_motivo'] or 'ninguno', 'se reporta', True),
        _control('V20', 'Registros conservados con algún nulo en campos críticos', 'informativo',
                 h['nulos_criticos_aceptados'], 'se reporta', True),
    ]
    return c


def tasas(controles, h):
    evaluados = [c for c in controles if c['severidad'] != 'informativo']
    aprobados = sum(c['aprobado'] for c in evaluados)
    return {
        'controles_evaluados': len(evaluados),
        'controles_aprobados': aprobados,
        'controles_fallidos': [c['id'] for c in evaluados if not c['aprobado']],
        'tasa_validaciones_pct': round(100 * aprobados / len(evaluados), 2),
        'registros_validos_pct': round(100 * h['aceptados'] / h['leidos'], 4) if h['leidos'] else 0.0,
    }


def escribir_log(ruta, metricas, controles, hechos):
    """Agrega al log .txt el bloque de una ejecución (registros, errores, tiempos y controles)."""
    ruta.parent.mkdir(parents=True, exist_ok=True)
    m = metricas
    lineas = [
        '=' * 78,
        f"PoC {m['herramienta'].upper()} — ejecución {m['ejecucion']} — {m['ts_inicio']} — entorno: {m['entorno']}",
        '=' * 78,
        f"Registros leídos:     {m['leidos']:,}",
        f"Registros conservados: {m['aceptados']:,}  ({m['registros_validos_pct']} % de los leídos)",
        f"Duplicados exactos eliminados (100 % de atributos iguales): {m['rechazados']:,}",
        f"Conservados con observación de calidad: {hechos['observaciones_por_motivo'] or 'ninguno'}",
        f"Conservados con nulos en campos críticos: {hechos['nulos_criticos_aceptados']}",
        f"Filas de incidentes con varios despachos: {hechos['filas_incidente_repetido']}",
        f"Errores de ejecución: {m['errores_ejecucion'] or 'ninguno'}",
        f"Tiempo total carga→exportación: {m['tiempo_total_s']:.3f} s   "
        f"(arranque de sesión aparte: {m['tiempo_inicio_sesion_s']:.3f} s; verificación de salidas: "
        f"{m['tiempo_verificacion_s']:.3f} s)",
        f"Throughput: {m['throughput_reg_min']:,.0f} registros/minuto",
        f"Memoria del sistema: base {m['mem_sis_base_pct']} % · pico muestreado {m['mem_sis_pico_pct']} %   "
        f"RSS proceso+hijos: base {m['rss_base_mb']} MB · pico {m['rss_pico_mb']} MB",
        f"Parquet detalle: {hechos['parquet_archivos']} archivos, {hechos['parquet_bytes']:,} bytes, "
        f"particiones {hechos['parquet_particiones']}",
        f"Tasa de éxito de validaciones: {m['tasa_validaciones_pct']} % "
        f"({m['controles_aprobados']}/{m['controles_evaluados']} controles bloqueantes+advertencia aprobados)",
        '', 'Controles:',
    ]
    for c in controles:
        estado = 'OK   ' if c['aprobado'] else 'FALLA'
        lineas.append(f"  [{estado}] {c['id']} ({c['severidad']}) {c['descripcion']}: valor={c['valor']} esperado={c['esperado']}")
    lineas.append('')
    with open(ruta, 'a', encoding='utf-8') as f:
        f.write('\n'.join(lineas) + '\n')

**Pipeline del PoC en PySpark (`pipeline_spark.py`).** Los 12 pasos descritos arriba.

In [ ]:
%%writefile /content/u2lib/src/emergencias123/pipeline_spark.py
"""PoC en PySpark: ingesta, tipado, estandarización, validación, enriquecimiento, agregados y Parquet.

Decisiones (ver la skill pipeline-pyspark-u2):
- Esquema explícito, todo texto, y tipado posterior con try_* porque Spark 4 activa ANSI por defecto
  (un cast inválido fallaría en lugar de dar nulo).
- Regla de limpieza (definida por el usuario): SOLO se eliminan los duplicados exactos (filas idénticas en el
  100 % de sus atributos originales; se conserva la primera aparición). Las copias eliminadas se escriben en
  `duplicados_eliminados/`, así que nada se descarta en silencio.
- Los nulos NO se eliminan ni se imputan. Los registros con problemas de calidad (fecha inválida, localidad sin
  dato, código inconsistente, edad fuera de rango…) se CONSERVAN y quedan marcados en `observacion_calidad`.
- El catálogo de tipos (23 filas, derivado y no oficial) se une con broadcast.
- El detalle se particiona por `periodo` (mes del archivo de origen). La variante periodo/localidad
  es configurable, pero genera muchos archivos pequeños con este volumen.
"""
from .catalogo import leer_catalogo
from .normalizacion import CON_TILDE, SIN_TILDE
from .validaciones import COLUMNAS_DETALLE, MOTIVOS, leer_localidades, prevalidar


def procesar(spark, cfg, salida, particionamiento=None):
    """Ejecuta el pipeline de la carga a la exportación y devuelve los hechos medidos."""
    from pyspark.sql import Window
    from pyspark.sql import functions as F
    from pyspark.sql.types import BooleanType, StringType, StructField, StructType

    esq, poc = cfg.esquema, cfg.poc
    hechos = prevalidar(cfg)
    rutas = [str(r) for r in cfg.rutas_csv if r.exists()]

    def norm(c):
        return F.upper(F.trim(F.regexp_replace(F.translate(c, CON_TILDE, SIN_TILDE), r'\s+', ' ')))

    def limpio(nombre):
        t = F.trim(F.col(nombre))
        return F.when(t.isNull() | t.isin(*esq['valores_nulos']), F.lit(None)).otherwise(t).alias(nombre)

    def mapa(d):
        return F.create_map(*[F.lit(x) for par in d.items() for x in par])

    # 1-2. Ingesta con esquema explícito + procedencia
    esquema = StructType([StructField(c, StringType(), True) for c in esq['columnas']]
                         + [StructField('_corrupt_record', StringType(), True)])
    crudo = (spark.read.csv(rutas, schema=esquema, sep=cfg.datos['separador'], header=True, encoding='UTF-8',
                            mode='PERMISSIVE', columnNameOfCorruptRecord='_corrupt_record')
             .withColumn('archivo_origen', F.col('_metadata.file_name')))

    # 3. Nulos explícitos (N/A y vacío -> null, sin imputar)
    df = crudo.select(*[limpio(c) for c in esq['columnas']], '_corrupt_record', 'archivo_origen')
    df = df.withColumn('periodo', mapa(cfg.periodo_por_archivo)[F.col('archivo_origen')])

    # 4-5. Tipos y estandarización
    localidades = leer_localidades(cfg)
    factores = {k: float(v) for k, v in poc['factores_edad'].items()}
    df = df.withColumns({
        'fecha_inicio': F.try_to_timestamp(F.col('FECHA_INICIO_DESPLAZAMIENTO_MOVIL'), F.lit(esq['formato_fecha_spark'])),
        'fecha_recepcion': F.try_to_timestamp(F.col('RECEPCION'), F.lit(esq['formato_fecha_spark'])),
        'codigo_localidad': F.col('CODIGO_LOCALIDAD').try_cast('int'),
        'edad': F.col('EDAD').try_cast('int'),
        'localidad_norm': norm(F.col('LOCALIDAD')),
        'unidad': norm(F.col('UNIDAD')),
        'genero': norm(F.col('GENERO')),
        'prioridad': norm(F.col('PRIORIDAD_FINAL')),
        'codigo_tipo': F.split(norm(F.col('TIPO_INCIDENTE')), ' - ', 2)[0],
    })
    df = df.withColumns({
        'edad_anios': F.col('edad') * mapa(factores)[F.col('unidad')],
        'paciente_completo': F.col('edad').isNotNull() & F.col('unidad').isNotNull() & F.col('genero').isNotNull(),
    })
    inicio_mes = F.to_timestamp(F.concat(F.col('periodo'), F.lit('-01 00:00:00')))
    fin_mes = F.add_months(inicio_mes, 1).cast('timestamp')
    df = df.withColumns({
        'en_mes_archivo': (F.col('fecha_inicio') >= inicio_mes) & (F.col('fecha_inicio') < fin_mes),
        'en_ventana': (F.col('fecha_inicio') >= inicio_mes)
                      & (F.col('fecha_inicio') < F.date_add(fin_mes, poc['tolerancia_dias_fin_mes']).cast('timestamp')),
    })

    # 6. Observaciones de calidad (la primera que aplique): el registro se CONSERVA, solo se marca
    codigo_esperado = mapa(localidades)[F.col('localidad_norm')]
    df = df.withColumn('observacion_calidad', (
        F.when(F.col('_corrupt_record').isNotNull(), 'registro_corrupto')
        .when(F.col('NUMERO_INCIDENTE').isNull(), 'numero_incidente_nulo')
        .when(F.col('fecha_inicio').isNull(), 'fecha_inicio_invalida')
        .when(F.col('localidad_norm').isNull() | (F.col('localidad_norm') == esq['localidad_sin_dato']), 'localidad_sin_dato')
        .when(~F.col('localidad_norm').isin(list(localidades)), 'localidad_fuera_catalogo')
        .when(F.col('codigo_localidad').isNull() | (F.col('codigo_localidad') != codigo_esperado),
              'codigo_localidad_inconsistente')
        .when(F.col('codigo_tipo').isNull(), 'tipo_incidente_nulo')
        .when(F.col('edad_anios').isNotNull() & ((F.col('edad_anios') < poc['edad_min_anios'])
                                                 | (F.col('edad_anios') > poc['edad_max_anios'])), 'edad_fuera_rango')
    ))

    # 7. Duplicados exactos: se ELIMINAN las copias (100 % de atributos iguales) y se conserva la primera aparición
    df = df.withColumn('_fila', F.monotonically_increasing_id())
    df = df.withColumns({
        'es_duplicado_exacto': F.row_number().over(Window.partitionBy(*esq['columnas']).orderBy('_fila')) > 1,
        'incidente_multidespacho': F.count(F.lit(1)).over(Window.partitionBy('NUMERO_INCIDENTE')) > 1,
    }).cache()

    unicos = ~F.col('es_duplicado_exacto')
    r = df.agg(  # una sola acción para todos los conteos de calidad de la entrada
        F.count(F.lit(1)).alias('leidos'),
        F.sum(unicos.cast('int')).alias('aceptados'),
        F.sum((F.col('FECHA_INICIO_DESPLAZAMIENTO_MOVIL').isNotNull() & F.col('fecha_inicio').isNull()).cast('int'))
        .alias('fechas_invalidas'),
        F.sum(F.col('es_duplicado_exacto').cast('int')).alias('duplicados_exactos'),
        F.sum((unicos & F.col('incidente_multidespacho')).cast('int')).alias('filas_incidente_repetido'),
        *[F.sum((unicos & (F.col('observacion_calidad') == m)).cast('int')).alias(f'motivo__{m}') for m in MOTIVOS],
    ).first().asDict()
    hechos['observaciones_por_motivo'] = {k.split('__', 1)[1]: int(v) for k, v in r.items()
                                          if k.startswith('motivo__') and v}
    hechos.update({k: int(v or 0) for k, v in r.items() if not k.startswith('motivo__')})
    hechos['rechazados'] = hechos['leidos'] - hechos['aceptados']  # = copias duplicadas exactas eliminadas
    hechos['rechazos_por_motivo'] = {'duplicado_exacto': hechos['rechazados']} if hechos['rechazados'] else {}
    hechos['inconsistencias_localidad'] = hechos['observaciones_por_motivo'].get('codigo_localidad_inconsistente', 0)

    # 8. Enriquecimiento con el catálogo derivado (broadcast; 23 filas)
    filas_cat = leer_catalogo(cfg.ruta_catalogo_tipos)
    catalogo = spark.createDataFrame(
        [(f['codigo_tipo'], f['descripcion_tipo'], f['codigo_explicito']) for f in filas_cat],
        StructType([StructField('codigo_tipo', StringType()), StructField('descripcion_tipo', StringType()),
                    StructField('codigo_explicito', BooleanType())]))
    hechos['catalogo_clave_unica'] = len(filas_cat) == len({f['codigo_tipo'] for f in filas_cat})
    aceptados = df.filter(unicos)
    hechos['filas_antes_join'] = hechos['aceptados']
    enriquecido = (aceptados.join(F.broadcast(catalogo), on='codigo_tipo', how='left')
                   .withColumn('codigo_reconocido', F.col('descripcion_tipo').isNotNull()))
    detalle = enriquecido.select(
        F.col('NUMERO_INCIDENTE').alias('numero_incidente'), 'fecha_inicio', 'fecha_recepcion', 'periodo',
        'archivo_origen', 'en_mes_archivo', 'codigo_localidad', 'localidad_norm', F.col('LOCALIDAD').alias('localidad'),
        'edad', 'unidad', 'edad_anios', 'genero', 'paciente_completo', F.col('TIPO_INCIDENTE').alias('tipo_incidente'),
        'codigo_tipo', 'descripcion_tipo', 'codigo_explicito', 'codigo_reconocido', 'prioridad',
        'observacion_calidad', 'incidente_multidespacho', 'en_ventana',
    ).cache()
    criticos = ['numero_incidente', 'fecha_inicio', 'codigo_localidad', 'localidad_norm', 'codigo_tipo']
    r = detalle.agg(
        F.count(F.lit(1)).alias('filas_despues_join'),
        F.sum((~F.col('codigo_reconocido')).cast('int')).alias('codigos_no_reconocidos'),
        F.sum((~F.col('en_ventana')).cast('int')).alias('fuera_ventana_aceptados'),
        F.sum((~F.col('en_mes_archivo')).cast('int')).alias('fuera_mes_aceptados'),
        F.sum(F.greatest(*[F.col(c).isNull().cast('int') for c in criticos])).alias('nulos_criticos_aceptados'),
        F.sum((F.greatest(*[F.col(c).isNull() for c in criticos]) & F.col('observacion_calidad').isNull()).cast('int'))
        .alias('nulos_criticos_sin_observacion'),
        F.sum((F.col('edad_anios').isNotNull() & ((F.col('edad_anios') < poc['edad_min_anios'])
                                                  | (F.col('edad_anios') > poc['edad_max_anios']))).cast('int'))
        .alias('edades_fuera_rango_aceptados'),
    ).first()
    hechos.update({k: int(r[k] or 0) for k in r.asDict()})
    detalle = detalle.drop('en_ventana')

    # 9. Agregados
    agregados = {  # el principal se guarda en caché: se suma (V12) y se escribe sin recomputarse
        'periodo_localidad_tipo': detalle.groupBy('periodo', 'localidad_norm', 'codigo_tipo', 'descripcion_tipo')
        .agg(F.count(F.lit(1)).alias('conteo')).cache(),
        'periodo_prioridad': detalle.groupBy('periodo', 'prioridad').agg(F.count(F.lit(1)).alias('conteo')),
        'diario': detalle.groupBy(F.to_date('fecha_inicio').alias('fecha')).agg(F.count(F.lit(1)).alias('conteo')),
    }
    hechos['suma_agregado_principal'] = int(agregados['periodo_localidad_tipo'].agg(F.sum('conteo')).first()[0] or 0)

    # 10. Exportación Parquet
    modo = particionamiento or poc['particionamiento']
    columnas_particion = ['periodo', 'localidad_norm'] if modo == 'periodo_localidad' else ['periodo']
    (detalle.repartition(*columnas_particion).sortWithinPartitions('localidad_norm', 'fecha_inicio')
     .write.mode('overwrite').partitionBy(*columnas_particion).parquet(str(salida / 'detalle')))
    for nombre, agg in agregados.items():
        agg.coalesce(1).write.mode('overwrite').parquet(str(salida / 'agregados' / nombre))
    (df.filter(F.col('es_duplicado_exacto'))
     .select(*esq['columnas'], 'archivo_origen', 'periodo')
     .coalesce(1).write.mode('overwrite').parquet(str(salida / 'duplicados_eliminados')))

    hechos['particionamiento'] = modo
    agregados['periodo_localidad_tipo'].unpersist()
    detalle.unpersist()
    df.unpersist()
    assert sorted(detalle.columns) == sorted(COLUMNAS_DETALLE), 'Esquema de detalle inesperado'
    return hechos

**Mismo alcance en Pandas (`pipeline_pandas.py`).** Reproduce las reglas del pipeline de Spark para comparar
en igualdad de condiciones (mismas entradas, mismas salidas en Drive).

In [ ]:
%%writefile /content/u2lib/src/emergencias123/pipeline_pandas.py
"""Mismo alcance del PoC implementado en Pandas, para comparar en igualdad de condiciones.

Reproduce las reglas de pipeline_spark.py: tipado sin imputación, observaciones de calidad (sin eliminar),
eliminación solo de duplicados exactos, enriquecimiento con el catálogo derivado, agregados y Parquet particionado por periodo.
"""
import shutil

import numpy as np
import pandas as pd

from .catalogo import leer_catalogo
from .normalizacion import CON_TILDE, SIN_TILDE
from .validaciones import COLUMNAS_DETALLE, leer_localidades, prevalidar

TILDES = str.maketrans(CON_TILDE, SIN_TILDE)


def _norm(serie):
    return serie.str.translate(TILDES).str.replace(r'\s+', ' ', regex=True).str.strip().str.upper()


def procesar(cfg, salida, particionamiento=None):
    esq, poc = cfg.esquema, cfg.poc
    hechos = prevalidar(cfg)

    # 1-3. Ingesta (todo texto), procedencia y nulos explícitos
    partes = []
    for ruta in cfg.rutas_csv:
        if ruta.exists():
            p = pd.read_csv(ruta, sep=cfg.datos['separador'], encoding=cfg.datos['codificacion'], dtype=str,
                            keep_default_na=False, na_values=[''])
            partes.append(p.assign(archivo_origen=ruta.name, periodo=cfg.periodo_por_archivo[ruta.name]))
    df = pd.concat(partes, ignore_index=True)
    for c in esq['columnas']:
        df[c] = df[c].str.strip().where(~df[c].str.strip().isin(esq['valores_nulos']))

    # 4-5. Tipos y estandarización
    localidades = leer_localidades(cfg)
    df['fecha_inicio'] = pd.to_datetime(df['FECHA_INICIO_DESPLAZAMIENTO_MOVIL'], format=esq['formato_fecha'], errors='coerce')
    df['fecha_recepcion'] = pd.to_datetime(df['RECEPCION'], format=esq['formato_fecha'], errors='coerce')
    df['codigo_localidad'] = pd.to_numeric(df['CODIGO_LOCALIDAD'], errors='coerce').astype('Int32')
    df['edad'] = pd.to_numeric(df['EDAD'], errors='coerce').astype('Int32')
    df['localidad_norm'] = _norm(df['LOCALIDAD'])
    df['unidad'] = _norm(df['UNIDAD'])
    df['genero'] = _norm(df['GENERO'])
    df['prioridad'] = _norm(df['PRIORIDAD_FINAL'])
    df['codigo_tipo'] = _norm(df['TIPO_INCIDENTE']).str.split(' - ', n=1).str[0]
    df['edad_anios'] = df['edad'].astype('Float64') * df['unidad'].map(poc['factores_edad']).astype('Float64')
    df['paciente_completo'] = df['edad'].notna() & df['unidad'].notna() & df['genero'].notna()
    inicio_mes = pd.to_datetime(df['periodo'] + '-01')
    fin_mes = inicio_mes + pd.DateOffset(months=1)
    df['en_mes_archivo'] = (df['fecha_inicio'] >= inicio_mes) & (df['fecha_inicio'] < fin_mes)
    en_ventana = (df['fecha_inicio'] >= inicio_mes) & (
        df['fecha_inicio'] < fin_mes + pd.Timedelta(days=poc['tolerancia_dias_fin_mes']))

    # 6. Observaciones de calidad (la primera que aplique): el registro se CONSERVA, solo se marca
    esperado = df['localidad_norm'].map(localidades)
    edad_mala = df['edad_anios'].notna() & ((df['edad_anios'] < poc['edad_min_anios'])
                                            | (df['edad_anios'] > poc['edad_max_anios']))
    condiciones = [
        pd.Series(False, index=df.index),
        df['NUMERO_INCIDENTE'].isna(),
        df['fecha_inicio'].isna(),
        df['localidad_norm'].isna() | (df['localidad_norm'] == esq['localidad_sin_dato']),
        ~df['localidad_norm'].isin(list(localidades)),
        df['codigo_localidad'].isna() | (df['codigo_localidad'] != esperado),
        df['codigo_tipo'].isna(),
        edad_mala,
    ]
    motivos = ['registro_corrupto', 'numero_incidente_nulo', 'fecha_inicio_invalida', 'localidad_sin_dato',
               'localidad_fuera_catalogo', 'codigo_localidad_inconsistente', 'tipo_incidente_nulo', 'edad_fuera_rango']
    df['observacion_calidad'] = np.select([c.fillna(True).to_numpy(dtype=bool) for c in condiciones], motivos, default=None)
    df['observacion_calidad'] = df['observacion_calidad'].where(df['observacion_calidad'].notna(), None)

    # 7. Duplicados exactos: se ELIMINAN las copias (100 % de atributos iguales); se conserva la primera aparición
    df['es_duplicado_exacto'] = df.duplicated(subset=esq['columnas'], keep='first')
    df['incidente_multidespacho'] = df.groupby('NUMERO_INCIDENTE', dropna=False)['NUMERO_INCIDENTE'].transform('size') > 1

    rechazo = df['es_duplicado_exacto']  # lo único que se elimina
    hechos.update({
        'leidos': len(df), 'aceptados': int((~rechazo).sum()), 'rechazados': int(rechazo.sum()),
        'fechas_invalidas': int((df['FECHA_INICIO_DESPLAZAMIENTO_MOVIL'].notna() & df['fecha_inicio'].isna()).sum()),
        'duplicados_exactos': int(df['es_duplicado_exacto'].sum()),
        'filas_incidente_repetido': int((~rechazo & df['incidente_multidespacho']).sum()),
        'observaciones_por_motivo': {k: int(v) for k, v in
                                     df.loc[~rechazo, 'observacion_calidad'].value_counts().items()},
    })
    hechos['rechazos_por_motivo'] = {'duplicado_exacto': hechos['rechazados']} if hechos['rechazados'] else {}
    hechos['inconsistencias_localidad'] = hechos['observaciones_por_motivo'].get('codigo_localidad_inconsistente', 0)

    # 8. Enriquecimiento con el catálogo derivado
    filas_cat = leer_catalogo(cfg.ruta_catalogo_tipos)
    catalogo = pd.DataFrame(filas_cat)[['codigo_tipo', 'descripcion_tipo', 'codigo_explicito']]
    hechos['catalogo_clave_unica'] = catalogo['codigo_tipo'].is_unique
    aceptados = df.loc[~rechazo].assign(en_ventana=en_ventana[~rechazo])
    hechos['filas_antes_join'] = len(aceptados)
    enriquecido = aceptados.merge(catalogo, on='codigo_tipo', how='left', validate='many_to_one')
    enriquecido['codigo_reconocido'] = enriquecido['descripcion_tipo'].notna()
    detalle = enriquecido.rename(columns={'NUMERO_INCIDENTE': 'numero_incidente', 'LOCALIDAD': 'localidad',
                                          'TIPO_INCIDENTE': 'tipo_incidente'})
    criticos = ['numero_incidente', 'fecha_inicio', 'codigo_localidad', 'localidad_norm', 'codigo_tipo']
    hechos.update({
        'filas_despues_join': len(detalle),
        'codigos_no_reconocidos': int((~detalle['codigo_reconocido']).sum()),
        'fuera_ventana_aceptados': int((detalle['fecha_inicio'].notna() & ~detalle['en_ventana']).sum()),
        'fuera_mes_aceptados': int((detalle['fecha_inicio'].notna() & ~detalle['en_mes_archivo']).sum()),
        'nulos_criticos_aceptados': int(detalle[criticos].isna().any(axis=1).sum()),
        'nulos_criticos_sin_observacion': int((detalle[criticos].isna().any(axis=1)
                                               & detalle['observacion_calidad'].isna()).sum()),
        'edades_fuera_rango_aceptados': int((detalle['edad_anios'].notna() & (
            (detalle['edad_anios'] < poc['edad_min_anios']) | (detalle['edad_anios'] > poc['edad_max_anios']))).sum()),
    })
    detalle = detalle[COLUMNAS_DETALLE]

    # 9. Agregados
    agregados = {
        'periodo_localidad_tipo': detalle.groupby(['periodo', 'localidad_norm', 'codigo_tipo', 'descripcion_tipo'],
                                                  dropna=False).size().reset_index(name='conteo'),
        'periodo_prioridad': detalle.groupby(['periodo', 'prioridad'], dropna=False).size().reset_index(name='conteo'),
        'diario': detalle.groupby(detalle['fecha_inicio'].dt.date.rename('fecha'), dropna=False).size().reset_index(name='conteo'),
    }
    hechos['suma_agregado_principal'] = int(agregados['periodo_localidad_tipo']['conteo'].sum())

    # 10. Exportación Parquet (mismo particionamiento que Spark)
    modo = particionamiento or poc['particionamiento']
    columnas_particion = ['periodo', 'localidad_norm'] if modo == 'periodo_localidad' else ['periodo']
    for sub in ('detalle', 'agregados', 'duplicados_eliminados'):
        shutil.rmtree(salida / sub, ignore_errors=True)
    (detalle.sort_values(['periodo', 'localidad_norm', 'fecha_inicio'])
     .to_parquet(salida / 'detalle', partition_cols=columnas_particion, index=False))
    for nombre, agg in agregados.items():
        (salida / 'agregados' / nombre).mkdir(parents=True, exist_ok=True)
        agg.to_parquet(salida / 'agregados' / nombre / 'parte-0.parquet', index=False)
    (salida / 'duplicados_eliminados').mkdir(parents=True, exist_ok=True)
    (df.loc[rechazo, esq['columnas'] + ['archivo_origen', 'periodo']]
     .to_parquet(salida / 'duplicados_eliminados' / 'parte-0.parquet', index=False))
    hechos['particionamiento'] = modo
    return hechos

**Ejecución y medición del PoC (`poc.py`).** Cada ejecución corre en un subproceso nuevo y registra el tiempo
total de la carga a la exportación (el arranque de la sesión aparte), la memoria, el throughput, las tasas, los
registros procesados y los errores. También genera los datos replicados del experimento de escalabilidad.

In [ ]:
%%writefile /content/u2lib/src/emergencias123/poc.py
"""Ejecución y medición del PoC (Spark) y de su equivalente en Pandas.

Cada ejecución corre en un subproceso nuevo (`python -m emergencias123.poc ...`) y registra:
tiempo total de la carga a la exportación (sin el arranque de la SparkSession, que se informa
aparte), memoria del sistema y RSS del proceso más sus hijos (JVM) muestreadas, throughput,
tasas de validación, registros procesados y errores. Las métricas se agregan a
`mediciones/poc.jsonl` y los controles al log `logs/validacion_poc_<herramienta>.txt`.
"""
import argparse
import json
import logging
import os
import subprocess
import sys
import time
import traceback
from datetime import datetime
from pathlib import Path

from .config import cargar_config
from .memoria import MemorySampler, medir
from .validaciones import escribir_log, evaluar, tasas, verificar_salidas

log = logging.getLogger(__name__)


def ejecutar_poc(herramienta, ejecucion, cfg, particionamiento=None, etiqueta=None):
    sufijo = f'_{etiqueta}' if etiqueta else ''
    salida = cfg.dir_salida('parquet', 'poc', herramienta, f'e{ejecucion}{sufijo}')
    metricas = {'herramienta': herramienta, 'ejecucion': ejecucion, 'etiqueta': etiqueta, 'entorno': cfg.entorno,
                'ts_inicio': datetime.now().isoformat(timespec='seconds'), 'ruta_salida': str(salida),
                'particionamiento': particionamiento or cfg.poc['particionamiento'], 'errores_ejecucion': []}
    hechos, spark = {}, None
    with MemorySampler(cfg.benchmark['intervalo_muestreo_s']) as s:
        try:
            if herramienta == 'spark':
                from .pipeline_spark import procesar
                from .spark_session import configuracion_spark, crear_sesion
                spark, t_ini, m_ini = medir(s, crear_sesion, cfg)
                metricas['config_spark'] = configuracion_spark(cfg)
                hechos, t_total, m_total = medir(s, procesar, spark, cfg, salida, particionamiento)
            else:
                from .pipeline_pandas import procesar
                t_ini, m_ini = 0.0, None
                hechos, t_total, m_total = medir(s, procesar, cfg, salida, particionamiento)
            verif, t_verif, _ = medir(s, verificar_salidas, salida)
            hechos.update(verif)
        except Exception as error:  # se registra y se reporta, no se oculta
            metricas['errores_ejecucion'].append(f'{type(error).__name__}: {error}')
            metricas['traza'] = traceback.format_exc()[-3000:]
        finally:
            if spark is not None:
                from .spark_session import detener_sesion
                detener_sesion(spark)
    if metricas['errores_ejecucion']:
        metricas['estado'] = 'error'
        return metricas, hechos, []
    controles = evaluar(hechos)
    metricas.update({
        'estado': 'ok',
        'tiempo_inicio_sesion_s': round(t_ini, 4),
        'tiempo_total_s': round(t_total, 4),
        'tiempo_verificacion_s': round(t_verif, 4),
        'tiempo_total_con_verificacion_s': round(t_total + t_verif, 4),
        **m_total,
        'leidos': hechos['leidos'], 'aceptados': hechos['aceptados'], 'rechazados': hechos['rechazados'],
        'throughput_reg_min': round(hechos['leidos'] / (t_total / 60), 1),
        **tasas(controles, hechos),
        'parquet_archivos': hechos['parquet_archivos'], 'parquet_bytes': hechos['parquet_bytes'],
        'rechazos_por_motivo': hechos['rechazos_por_motivo'],  # solo copias duplicadas exactas eliminadas
        'observaciones_por_motivo': hechos['observaciones_por_motivo'],  # conservados con anomalía marcada
        'nulos_criticos_aceptados': hechos['nulos_criticos_aceptados'],
        'duplicados_exactos': hechos['duplicados_exactos'],
        'filas_incidente_repetido': hechos['filas_incidente_repetido'],
        'fuera_mes_aceptados': hechos['fuera_mes_aceptados'],
    })
    metricas['errores'] = len(metricas['controles_fallidos'])
    return metricas, hechos, controles


def ejecutar_subproceso(herramienta, ejecucion, particionamiento=None, etiqueta=None, timeout=3600, entorno_extra=None):
    src = str(Path(__file__).resolve().parents[1])
    env = dict(os.environ, PYTHONPATH=src + os.pathsep + os.environ.get('PYTHONPATH', ''), **(entorno_extra or {}))
    cmd = [sys.executable, '-m', 'emergencias123.poc', '--herramienta', herramienta, '--ejecucion', str(ejecucion)]
    if particionamiento:
        cmd += ['--particionamiento', particionamiento]
    if etiqueta:
        cmd += ['--etiqueta', etiqueta]
    t0 = time.perf_counter()
    r = subprocess.run(cmd, env=env, capture_output=True, text=True, timeout=timeout)
    return {'herramienta': herramienta, 'ejecucion': ejecucion, 'etiqueta': etiqueta, 'codigo_salida': r.returncode,
            'segundos_proceso': round(time.perf_counter() - t0, 2), 'salida': r.stdout[-800:],
            'stderr_final': r.stderr[-2000:] if r.returncode else ''}


def ejecutar_poc_completo(cfg, herramientas=('spark', 'pandas'), ejecuciones=None, al_avanzar=None):
    """N ejecuciones por herramienta, alternando el orden en cada ronda."""
    ejecuciones = ejecuciones or cfg.poc['ejecuciones']
    resultados = []
    for e in range(1, ejecuciones + 1):
        orden = list(herramientas) if e % 2 else list(reversed(herramientas))
        for h in orden:
            res = ejecutar_subproceso(h, e)
            resultados.append(res)
            if al_avanzar:
                al_avanzar(res)
    return resultados


def replicar_datos(cfg, factor, destino):
    """Crea CSV con el contenido de cada mes repetido `factor` veces (DATOS SINTÉTICOS por replicación).

    Sirve para medir cómo crecen tiempo y memoria con el volumen (x4 ≈ 1 año, x8 ≈ 2 años de filas).
    La réplica k (k ≥ 1) agrega el sufijo `-R<k>` a NUMERO_INCIDENTE (primera columna): así las réplicas NO son
    duplicados exactos de la original (si lo fueran, la regla de limpieza las eliminaría y el volumen procesado
    no crecería). Los duplicados exactos propios de los datos se repiten dentro de cada réplica.
    """
    destino.mkdir(parents=True, exist_ok=True)
    for ruta in cfg.rutas_csv:
        crudo = ruta.read_bytes()
        bom = b'\xef\xbb\xbf' if crudo.startswith(b'\xef\xbb\xbf') else b''
        encabezado, _, cuerpo = crudo[len(bom):].partition(b'\n')
        lineas = [ln for ln in cuerpo.split(b'\n') if ln.strip()]
        partes = [bom + encabezado + b'\n']
        for k in range(factor):
            if k == 0:
                partes.extend(ln + b'\n' for ln in lineas)
            else:
                sufijo = f'-R{k}'.encode()
                for ln in lineas:
                    clave, sep, resto = ln.partition(b';')
                    partes.append(clave + sufijo + sep + resto + b'\n')
        (destino / ruta.name).write_bytes(b''.join(partes))
    return destino


def ejecutar_escalabilidad(cfg, factores=(4, 8), base=Path('/content/u2_escala'), herramientas=('spark', 'pandas'),
                           al_avanzar=None):
    """Mide el PoC con volumen replicado; las métricas se agregan a mediciones/escalabilidad.jsonl de cfg."""
    import shutil
    resultados = []
    for factor in factores:
        datos = replicar_datos(cfg, factor, base / f'datos_x{factor}')
        salida = base / f'resultados_x{factor}'
        shutil.rmtree(salida, ignore_errors=True)
        for h in herramientas:
            res = ejecutar_subproceso(h, 1, etiqueta=f'replica_x{factor}',
                                      entorno_extra={'U2_DIR_DATOS': str(datos), 'U2_DIR_RESULTADOS': str(salida)})
            res['factor'] = factor
            resultados.append(res)
            if al_avanzar:
                al_avanzar(res)
        jsonl = salida / 'mediciones' / 'poc.jsonl'
        if jsonl.exists():
            with open(cfg.dir_salida('mediciones') / 'escalabilidad.jsonl', 'a', encoding='utf-8') as f:
                for linea in jsonl.read_text(encoding='utf-8').splitlines():
                    registro = json.loads(linea)
                    registro.update(factor_replica=factor, datos='sinteticos_replicados')
                    f.write(json.dumps(registro, ensure_ascii=False) + '\n')
    return resultados


def main():
    p = argparse.ArgumentParser(description='Una ejecución medida del PoC en este proceso.')
    p.add_argument('--herramienta', required=True, choices=['spark', 'pandas'])
    p.add_argument('--ejecucion', type=int, required=True)
    p.add_argument('--particionamiento', choices=['periodo', 'periodo_localidad'])
    p.add_argument('--etiqueta')
    a = p.parse_args()
    logging.basicConfig(level=logging.WARNING)
    cfg = cargar_config()
    metricas, hechos, controles = ejecutar_poc(a.herramienta, a.ejecucion, cfg, a.particionamiento, a.etiqueta)
    metricas['controles'] = controles
    ruta = cfg.dir_salida('mediciones') / 'poc.jsonl'
    with open(ruta, 'a', encoding='utf-8') as f:
        f.write(json.dumps(metricas, ensure_ascii=False, default=str) + '\n')
    if metricas['estado'] == 'ok':
        nombre = f'validacion_poc_{a.herramienta}{"_" + a.etiqueta if a.etiqueta else ""}.txt'
        escribir_log(cfg.dir_salida('logs') / nombre, metricas, controles, hechos)
        print(json.dumps({k: metricas[k] for k in ('herramienta', 'ejecucion', 'tiempo_total_s', 'mem_sis_pico_pct',
                                                   'rss_pico_mb', 'throughput_reg_min', 'tasa_validaciones_pct',
                                                   'registros_validos_pct', 'aceptados', 'rechazados')}, ensure_ascii=False))
    else:
        print(metricas.get('traza', ''), file=sys.stderr)
        sys.exit(1)


if __name__ == '__main__':
    main()

**Diagnósticos de optimización (`optimizacion.py`).**

In [ ]:
%%writefile /content/u2lib/src/emergencias123/optimizacion.py
"""Diagnósticos de optimización y calidad del PoC, trasladados de los notebooks de ejemplo del curso (Formatos/).

- Planes de ejecución (explain): evidencia de poda de particiones, predicados empujados al Parquet y broadcast join.
- Comparación de formatos CSV, JSON y Parquet con los datos del PoC: tamaño en disco y tiempo de lectura.
- Perfil de nulos por columna (en una sola agregación, no una acción por columna como en el ejemplo).
- Valores atípicos por rango intercuartílico (IQR), solo informativo: no se imputa ni se elimina nada.
Se ejecuta en Colab desde el notebook 02.
"""
import shutil
import statistics
import time
from pathlib import Path


def _plan(df):
    """Texto del plan físico: se captura la salida de la API pública df.explain(mode='formatted')."""
    import contextlib
    import io

    buffer = io.StringIO()
    with contextlib.redirect_stdout(buffer):
        df.explain(mode='formatted')
    return buffer.getvalue()


def planes_ejecucion(spark, ruta_detalle, ruta_catalogo):
    """Dos consultas representativas y lo que su plan físico demuestra."""
    from pyspark.sql import functions as F

    detalle = spark.read.parquet(str(ruta_detalle))
    consulta = (detalle.filter((F.col('periodo') == '2026-08') & (F.col('localidad_norm') == 'USAQUEN'))
                .groupBy('codigo_tipo').count())
    plan_filtro = _plan(consulta)
    catalogo = spark.read.csv(str(ruta_catalogo), header=True).select('codigo_tipo', F.col('descripcion_tipo').alias('desc_cat'))
    union = detalle.join(F.broadcast(catalogo), on='codigo_tipo', how='left').groupBy('desc_cat').count()
    plan_union = _plan(union)
    return {
        'plan_filtro': plan_filtro,
        'plan_union': plan_union,
        'poda_particiones': 'PartitionFilters' in plan_filtro and 'periodo' in plan_filtro.split('PartitionFilters')[1][:200],
        'predicado_empujado': 'PushedFilters' in plan_filtro and 'localidad_norm' in plan_filtro.split('PushedFilters')[1][:300],
        'broadcast_join': 'BroadcastHashJoin' in plan_union,
    }


def _tamano_mb(ruta):
    ruta = Path(ruta)
    archivos = [ruta] if ruta.is_file() else [p for p in ruta.rglob('*') if p.is_file() and not p.name.startswith(('.', '_'))]
    return sum(p.stat().st_size for p in archivos) / 2 ** 20


def comparar_formatos(spark, ruta_detalle, directorio_tmp, repeticiones=3):
    """Escribe el mismo detalle en CSV, JSON y Parquet y mide tamaño y lectura + agregación (mediana de N)."""
    from pyspark.sql import functions as F

    tmp = Path(directorio_tmp)
    shutil.rmtree(tmp, ignore_errors=True)
    detalle = spark.read.parquet(str(ruta_detalle)).cache()
    n = detalle.count()
    rutas = {'CSV': tmp / 'csv', 'JSON': tmp / 'json', 'Parquet': tmp / 'parquet'}
    detalle.coalesce(1).write.mode('overwrite').option('header', True).csv(str(rutas['CSV']))
    detalle.coalesce(1).write.mode('overwrite').json(str(rutas['JSON']))
    detalle.coalesce(1).write.mode('overwrite').parquet(str(rutas['Parquet']))
    detalle.unpersist()
    lectores = {
        'CSV': lambda: spark.read.option('header', True).option('inferSchema', True).csv(str(rutas['CSV'])),
        'JSON': lambda: spark.read.json(str(rutas['JSON'])),
        'Parquet': lambda: spark.read.parquet(str(rutas['Parquet'])),
    }
    resultados = []
    for formato, leer in lectores.items():
        tiempos = []
        for _ in range(repeticiones):
            t0 = time.perf_counter_ns()
            filas = leer().groupBy('codigo_tipo').agg(F.count(F.lit(1)).alias('n')).agg(F.sum('n')).first()[0]
            tiempos.append((time.perf_counter_ns() - t0) / 1e9)
        resultados.append({'formato': formato, 'mb': round(_tamano_mb(rutas[formato]), 3),
                           'lectura_agregacion_s': round(statistics.median(tiempos), 3), 'filas': int(filas)})
    base = next(r for r in resultados if r['formato'] == 'CSV')
    for r in resultados:
        r['tamano_vs_csv'] = round(r['mb'] / base['mb'], 3)
    shutil.rmtree(tmp, ignore_errors=True)
    return {'registros': n, 'repeticiones': repeticiones, 'formatos': resultados}


def perfil_nulos(df):
    """Nulos por columna con UNA sola agregación (el ejemplo del curso lanza una acción por columna)."""
    from pyspark.sql import functions as F

    fila = df.agg(F.count(F.lit(1)).alias('__total'),
                  *[F.sum(F.col(c).isNull().cast('int')).alias(c) for c in df.columns]).first().asDict()
    total = fila.pop('__total')
    return [{'columna': c, 'nulos': int(v or 0), 'porcentaje': round(100 * (v or 0) / total, 2)}
            for c, v in fila.items()]


def atipicos_iqr(df, columna, error_relativo=0.01):
    """Límites IQR (1.5 × IQR) con approxQuantile y conteo de valores fuera de ellos; no modifica los datos."""
    from pyspark.sql import functions as F

    q1, q3 = df.filter(F.col(columna).isNotNull()).approxQuantile(columna, [0.25, 0.75], error_relativo)
    iqr = q3 - q1
    inferior, superior = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    fuera = df.filter((F.col(columna) < inferior) | (F.col(columna) > superior)).count()
    return {'columna': columna, 'q1': q1, 'q3': q3, 'iqr': iqr, 'limite_inferior': inferior,
            'limite_superior': superior, 'atipicos': int(fuera)}

**Datos sintéticos de prueba y valores esperados (`conftest.py`).**

In [ ]:
%%writefile /content/u2lib/tests/conftest.py
import sys
from pathlib import Path

import pytest

RAIZ = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(RAIZ / 'src'))

from emergencias123.config import cargar_config  # noqa: E402

ENCABEZADO = ('NUMERO_INCIDENTE;FECHA_INICIO_DESPLAZAMIENTO_MOVIL;CODIGO_LOCALIDAD;LOCALIDAD;EDAD;UNIDAD;'
              'GENERO;TIPO_INCIDENTE;PRIORIDAD_FINAL;RECEPCION')
# Datos sintéticos de prueba (no son registros reales): cada fila ejercita una regla del pipeline.
FIXTURE = {
    'llamadas123_julio_2026.csv': [
        'CRU-1;2026-07-01 00:08:54;14;LOS MÁRTIRES;1;Años;MASCULINO;ACOMPAÑAMIENTO A EVENTO;Baja;',
        'CRU-2;2026-07-02 10:00:00;1;USAQUÉN;42;Años;FEMENINO;ENFERMO;Media;2026-07-02 11:00:00',
        'CRU-3;2026-07-03 10:00:00;2;CHAPINERO;N/A;N/A;N/A;AMESUI  - AMENAZA DE SUICIDIO;Alta;',
        'CRU-2;2026-07-02 10:00:00;1;USAQUÉN;42;Años;FEMENINO;ENFERMO;Media;2026-07-02 11:00:00',
        'CRU-4;fecha-mala;1;USAQUÉN;30;Años;MASCULINO;INTSUI - INTENTO DE SUICIDIO;Alta;',
        'CRU-5;2026-07-05 10:00:00;SIN_D;SIN_D;N/A;N/A;N/A;ENFERMO;Baja;',
        'CRU-6;2026-07-06 10:00:00;3;USAQUÉN;30;Años;MASCULINO;ENFERMO;Baja;',
        'CRU-7;2026-07-07 10:00:00;1;USAQUÉN;150;Años;MASCULINO;ENFERMO;Baja;',
        'CRU-8;2026-08-01 00:30:00;2;CHAPINERO;5;Meses;FEMENINO;CONVULSIÓN - CONVULSIÓN;Critica;',
    ],
    'llamadaslinea123agosto2026.csv': [
        'CRU-2;2026-08-02 10:00:00;1;USAQUÉN;42;Años;FEMENINO;ENFERMO;Media;',
        'CRU-9;2026-08-03 10:00:00;10;ENGATIVÁ;20;Años;MASCULINO;HERIDO - HERIDOS ACCIDENTALES;Alta;',
    ],
    'llamadaslinea123Septiembre.csv': [
        'CRU-10;2026-09-10 10:00:00;2;CHAPINERO;33;Años;FEMENINO;CODNUEVO - TIPO NUEVO;Alta;',
    ],
}
# Regla de limpieza: solo se elimina la copia duplicada exacta (CRU-2 repetido en julio). Las demás anomalías
# se conservan y quedan marcadas en `observacion_calidad`.
ESPERADO = {'leidos': 12, 'aceptados': 11, 'rechazados': 1, 'duplicados_exactos': 1, 'filas_incidente_repetido': 2,
            'codigos_no_reconocidos': 1, 'fuera_mes_aceptados': 1, 'fuera_ventana_aceptados': 0,
            'rechazos_por_motivo': {'duplicado_exacto': 1},
            'observaciones_por_motivo': {'fecha_inicio_invalida': 1, 'localidad_sin_dato': 1,
                                         'codigo_localidad_inconsistente': 1, 'edad_fuera_rango': 1},
            'nulos_criticos_aceptados': 2, 'nulos_criticos_sin_observacion': 0,
            'filtrado_benchmark': 9}


def pytest_configure(config):
    config.addinivalue_line('markers', 'spark: requiere PySpark con JVM (runtime de Google Colab)')


def pytest_collection_modifyitems(config, items):
    if not sys.platform.startswith('linux'):  # el proyecto se ejecuta en el runtime Linux de Colab
        omitir = pytest.mark.skip(reason='Las pruebas de Spark se ejecutan en el runtime de Google Colab')
        for item in items:
            if 'spark' in item.keywords:
                item.add_marker(omitir)


@pytest.fixture
def cfg_fixture(tmp_path, monkeypatch):
    datos = tmp_path / 'datos'
    datos.mkdir()
    for nombre, filas in FIXTURE.items():
        (datos / nombre).write_bytes(('﻿' + '\r\n'.join([ENCABEZADO, *filas]) + '\r\n').encode('utf-8'))
    monkeypatch.setenv('U2_DIR_DATOS', str(datos))
    return cargar_config(entorno='local', dir_resultados=tmp_path / 'resultados')


@pytest.fixture
def cfg_real(tmp_path):
    cfg = cargar_config(dir_resultados=tmp_path / 'resultados')
    if not all(r.exists() for r in cfg.rutas_csv):
        pytest.skip('CSV reales no disponibles en este entorno')
    return cfg

**Normalización y catálogo.**

In [ ]:
%%writefile /content/u2lib/tests/test_normalizacion_catalogo.py
import csv

import pandas as pd
import polars as pl
import pytest

from emergencias123 import ops_pandas, ops_polars
from emergencias123.catalogo import derivar_catalogo, leer_catalogo
from emergencias123.loc import loc_por_herramienta
from emergencias123.normalizacion import extraer_codigo_tipo, extraer_descripcion_tipo, normalizar_texto


@pytest.mark.parametrize('entrada, esperado', [
    ('  Usaquén ', 'USAQUEN'), ('LOS MÁRTIRES', 'LOS MARTIRES'), ('ANTONIO NARIÑO', 'ANTONIO NARINO'),
    ('', None), (None, None), ('a  b\tc', 'A B C'),
])
def test_normalizar_texto(entrada, esperado):
    assert normalizar_texto(entrada) == esperado


@pytest.mark.parametrize('entrada, codigo, descripcion', [
    ('INTSUI - INTENTO DE SUICIDIO', 'INTSUI', 'INTENTO DE SUICIDIO'),
    ('AMESUI  - AMENAZA DE SUICIDIO', 'AMESUI', 'AMENAZA DE SUICIDIO'),
    ('CONVULSIÓN - CONVULSIÓN', 'CONVULSION', 'CONVULSIÓN'),
    ('ENFERMO', 'ENFERMO', 'ENFERMO'),
    ('ELECTROCUC - ELECTROCUCIÓN / RESCATE', 'ELECTROCUC', 'ELECTROCUCIÓN / RESCATE'),
])
def test_extraer_codigo_y_descripcion(entrada, codigo, descripcion):
    assert extraer_codigo_tipo(entrada) == codigo
    assert extraer_descripcion_tipo(entrada) == descripcion


def _valores_reales(cfg, columna):
    valores = set()
    for ruta in cfg.rutas_csv:
        with open(ruta, encoding=cfg.datos['codificacion'], newline='') as f:
            valores |= {r[columna] for r in csv.DictReader(f, delimiter=';')}
    return sorted(valores)


def test_vectorizado_pandas_polars_igual_a_referencia(cfg_real):
    """Las versiones vectorizadas reproducen exactamente la función de referencia en los datos reales."""
    for col in ('LOCALIDAD', 'TIPO_INCIDENTE'):
        valores = _valores_reales(cfg_real, col)
        ref = [normalizar_texto(v) for v in valores]
        assert ops_pandas._normalizar(pd.Series(valores)).tolist() == ref
        assert pl.DataFrame({col: valores}).select(ops_polars._normalizar(col))[col].to_list() == ref
        assert all(r.isascii() for r in ref), 'quedan caracteres no ASCII sin mapear'


def test_catalogo_real_clave_unica_y_cobertura(cfg_real):
    catalogo = leer_catalogo(cfg_real.ruta_catalogo_tipos)
    assert len(catalogo) == 23
    codigos = {c['codigo_tipo'] for c in catalogo}
    assert {extraer_codigo_tipo(v) for v in _valores_reales(cfg_real, 'TIPO_INCIDENTE')} <= codigos
    assert all('no oficial' in c['fuente'] for c in catalogo)


def test_catalogo_ambiguo_falla():
    with pytest.raises(ValueError):
        derivar_catalogo(['ABC - UNO', 'ABC - DOS'])


def test_loc_contadas_para_todas_las_operaciones():
    loc = loc_por_herramienta()
    for herramienta, conteos in loc.items():
        assert set(conteos) == {'carga', 'filtrado', 'agregacion', 'join', 'exportacion'}, herramienta
        assert all(n > 0 for n in conteos.values()), herramienta

**Adquisición de datos (lógica adaptada de la Unidad 1).**

In [ ]:
%%writefile /content/u2lib/tests/test_adquisicion.py
"""Adquisición de datos (lógica adaptada de la Unidad 1): carpeta del proyecto → carpeta alterna → portal."""
import dataclasses

import pytest

from emergencias123.adquisicion import asegurar_datos, sha256


def _cfg_con(cfg, dir_datos, alternos, huellas):
    datos = dict(cfg.datos, archivos=[dict(a, sha256=huellas[a['nombre']], url=None) for a in cfg.datos['archivos']])
    return dataclasses.replace(cfg, dir_datos=dir_datos, datos_alternos=tuple(alternos), datos=datos)


def test_copia_desde_carpeta_alterna_y_verifica_huella(cfg_fixture, tmp_path):
    huellas = {p.name: sha256(p) for p in cfg_fixture.rutas_csv}
    cfg = _cfg_con(cfg_fixture, tmp_path / 'vacia', [cfg_fixture.dir_datos], huellas)
    r = asegurar_datos(cfg, permitir_descarga=False)
    assert all(v['origen'].startswith('copiado de') for v in r.values())
    assert all((tmp_path / 'vacia' / n).exists() for n in huellas)
    r2 = asegurar_datos(cfg, permitir_descarga=False)  # segunda vez: ya están en la carpeta del proyecto
    assert all(v['origen'] == 'carpeta del proyecto' for v in r2.values())


def test_huella_distinta_detiene_sin_corregir_en_silencio(cfg_fixture):
    huellas = {p.name: '0' * 64 for p in cfg_fixture.rutas_csv}
    with pytest.raises(RuntimeError, match='SHA-256'):
        asegurar_datos(_cfg_con(cfg_fixture, cfg_fixture.dir_datos, [], huellas), permitir_descarga=False)


def test_config_real_tiene_huella_y_url_por_archivo(cfg_real):
    for a in cfg_real.datos['archivos']:
        assert len(a['sha256']) == 64 and a['url'].startswith('https://datosabiertos.bogota.gov.co/')
    assert asegurar_datos(cfg_real, permitir_descarga=False)  # los CSV del proyecto tienen la huella esperada


def test_encuentra_en_carpeta_alterna_con_otro_nombre_por_huella(cfg_fixture, tmp_path):
    alterna = tmp_path / 'u1'
    alterna.mkdir()
    huellas = {}
    for i, p in enumerate(cfg_fixture.rutas_csv):
        huellas[p.name] = sha256(p)
        (alterna / f'otro_nombre_{i}.csv').write_bytes(p.read_bytes())
    cfg = _cfg_con(cfg_fixture, tmp_path / 'proyecto', [alterna], huellas)
    r = asegurar_datos(cfg, permitir_descarga=False)
    assert all(v['origen'].startswith('copiado de') for v in r.values())
    assert all((tmp_path / 'proyecto' / n).exists() for n in huellas)

**Benchmark y equivalencia entre herramientas.**

In [ ]:
%%writefile /content/u2lib/tests/test_benchmark_equivalencia.py
from pathlib import Path

import pytest

from conftest import ESPERADO
from emergencias123.benchmark import OPERACIONES, measure_performance, plan_benchmark
from emergencias123.equivalencia import comparar

HERRAMIENTAS_LOCALES = ['pandas', 'polars']


def _ejecutar(cfg, herramientas, modo):
    rutas = {}
    for h in herramientas:
        regs = measure_performance(h, 1, modo, False, cfg)
        assert all(r['estado'] == 'ok' for r in regs), regs[-1].get('error')
        ops = {r['operacion']: r for r in regs}
        esperadas = (OPERACIONES if modo == 'por_operacion' else ['extremo_a_extremo']) + ['total']
        assert set(esperadas) <= set(ops)
        for r in regs:
            assert r['mem_sis_pico_pct'] >= r['mem_sis_base_pct'] and r['rss_pico_mb'] >= r['rss_base_mb']
            assert r['n_muestras'] >= 1 and r['segundos'] >= 0
        if modo == 'por_operacion':
            assert ops['join']['filas_entrada'] == ops['join']['filas_salida'], 'el join multiplicó filas'
        rutas[h] = Path(regs[0]['ruta_salida'])
    return rutas


@pytest.mark.parametrize('modo', ['por_operacion', 'extremo_a_extremo', 'extremo_a_extremo_caliente'])
def test_equivalencia_fixture_pandas_polars(cfg_fixture, modo):
    res = comparar(_ejecutar(cfg_fixture, HERRAMIENTAS_LOCALES, modo))
    assert res['equivalentes'], res['criterios']
    assert res['resumenes']['pandas']['filas'] == ESPERADO['filtrado_benchmark']


def test_equivalencia_datos_reales_pandas_polars(cfg_real):
    res = comparar(_ejecutar(cfg_real, HERRAMIENTAS_LOCALES, 'por_operacion'))
    assert res['equivalentes'], res['criterios']
    assert res['resumenes']['pandas']['filas'] == 1103 + 659


@pytest.mark.spark
@pytest.mark.parametrize('modo', ['por_operacion', 'extremo_a_extremo', 'extremo_a_extremo_caliente'])
def test_equivalencia_tres_herramientas(cfg_real, modo):
    res = comparar(_ejecutar(cfg_real, ['pandas', 'polars', 'spark'], modo))
    assert res['equivalentes'], res['criterios']


def test_plan_rota_orden_y_separa_calentamiento(cfg_real):
    plan = plan_benchmark(cfg_real, modos=['por_operacion'], repeticiones=3)
    calentamientos = [p for p in plan if p[3]]
    medidas = [p for p in plan if not p[3]]
    assert len(calentamientos) == 3 and len(medidas) == 9
    primeras = [medidas[i * 3][0] for i in range(3)]
    assert primeras == ['pandas', 'polars', 'spark']

**Reglas del pipeline, tasas y replicación.**

In [ ]:
%%writefile /content/u2lib/tests/test_pipeline_validaciones.py
import pyarrow.dataset as ds
import pytest

from conftest import ESPERADO
from emergencias123.poc import ejecutar_poc
from emergencias123.validaciones import COLUMNAS_DETALLE, tasas


def _verificar_hechos(metricas, hechos, controles):
    assert metricas['estado'] == 'ok', metricas.get('traza')
    for clave in ('leidos', 'aceptados', 'rechazados', 'duplicados_exactos', 'filas_incidente_repetido',
                  'codigos_no_reconocidos', 'fuera_mes_aceptados', 'fuera_ventana_aceptados', 'rechazos_por_motivo',
                  'observaciones_por_motivo', 'nulos_criticos_aceptados', 'nulos_criticos_sin_observacion'):
        assert hechos[clave] == ESPERADO[clave], clave
    # El fixture provoca a propósito 4 advertencias: fecha no parseable (V05), código nuevo no reconocido (V08),
    # código de localidad inconsistente (V10) y edad fuera de rango (V11). Esos registros se CONSERVAN marcados.
    fallidos = {c['id'] for c in controles if not c['aprobado']}
    assert fallidos == {'V05', 'V08', 'V10', 'V11'}
    assert metricas['tasa_validaciones_pct'] == round(100 * 12 / 16, 2)
    assert metricas['registros_validos_pct'] == round(100 * 11 / 12, 4)
    detalle = ds.dataset(metricas['ruta_salida'] + '/detalle', format='parquet', partitioning='hive').to_table()
    assert sorted(detalle.column_names) == sorted(COLUMNAS_DETALLE)
    assert detalle.num_rows == ESPERADO['aceptados']
    observaciones = [o for o in detalle.column('observacion_calidad').to_pylist() if o]
    assert len(observaciones) == 4, 'las anomalías se conservan marcadas, no se eliminan'
    duplicados = ds.dataset(metricas['ruta_salida'] + '/duplicados_eliminados', format='parquet').to_table()
    assert duplicados.num_rows == ESPERADO['rechazados'], 'la copia eliminada queda registrada aparte'


def test_pipeline_pandas_reglas(cfg_fixture):
    _verificar_hechos(*ejecutar_poc('pandas', 1, cfg_fixture))


@pytest.mark.spark
def test_pipeline_spark_reglas(cfg_fixture):
    _verificar_hechos(*ejecutar_poc('spark', 1, cfg_fixture))


@pytest.mark.spark
def test_pipeline_spark_y_pandas_coinciden_en_datos_reales(cfg_real):
    m_s, h_s, _ = ejecutar_poc('spark', 1, cfg_real)
    m_p, h_p, _ = ejecutar_poc('pandas', 1, cfg_real)
    for clave in ('leidos', 'aceptados', 'rechazados', 'rechazos_por_motivo', 'duplicados_exactos',
                  'observaciones_por_motivo', 'nulos_criticos_aceptados', 'filas_incidente_repetido',
                  'parquet_conteo', 'suma_agregado_principal'):
        assert h_s[clave] == h_p[clave], clave
    # Solo se eliminan las 44 copias duplicadas exactas de los datos reales; los nulos se conservan.
    assert h_s['leidos'] - h_s['aceptados'] == h_s['duplicados_exactos'] == 44
    assert m_s['tasa_validaciones_pct'] == m_p['tasa_validaciones_pct'] == 100.0


def test_formula_tasas_excluye_informativos():
    controles = [{'id': 'A', 'severidad': 'bloqueante', 'aprobado': True},
                 {'id': 'B', 'severidad': 'advertencia', 'aprobado': False},
                 {'id': 'C', 'severidad': 'informativo', 'aprobado': True},
                 {'id': 'D', 'severidad': 'bloqueante', 'aprobado': True}]
    t = tasas(controles, {'aceptados': 90, 'leidos': 100})
    assert t['tasa_validaciones_pct'] == round(100 * 2 / 3, 2)
    assert t['controles_fallidos'] == ['B']
    assert t['registros_validos_pct'] == 90.0


def test_replicacion_escala_el_volumen_sin_crear_duplicados_exactos(cfg_fixture, tmp_path, monkeypatch):
    """Las réplicas sintéticas del experimento de escalabilidad no deben eliminarse como duplicados exactos."""
    from emergencias123.config import cargar_config
    from emergencias123.poc import replicar_datos
    destino = replicar_datos(cfg_fixture, 3, tmp_path / 'x3')
    monkeypatch.setenv('U2_DIR_DATOS', str(destino))
    cfg3 = cargar_config(entorno='local', dir_resultados=tmp_path / 'res_x3')
    metricas, hechos, _ = ejecutar_poc('pandas', 1, cfg3)
    assert metricas['estado'] == 'ok', metricas.get('traza')
    assert hechos['leidos'] == 3 * ESPERADO['leidos']
    assert hechos['aceptados'] == 3 * ESPERADO['aceptados']
    assert hechos['duplicados_exactos'] == 3 * ESPERADO['duplicados_exactos']

**Diagnósticos de optimización.**

In [ ]:
%%writefile /content/u2lib/tests/test_optimizacion.py
from pathlib import Path

import pytest

from emergencias123.poc import ejecutar_poc


@pytest.mark.spark
def test_planes_formatos_nulos_atipicos(cfg_fixture, tmp_path):
    from emergencias123.optimizacion import atipicos_iqr, comparar_formatos, perfil_nulos, planes_ejecucion
    from emergencias123.spark_session import crear_sesion, detener_sesion

    # Parquet de detalle escrito por Spark (el que analiza el notebook). El de Pandas usa marcas de tiempo en
    # nanosegundos, que Spark 4 no sabe leer (PARQUET_TYPE_ILLEGAL), así que no sirve como entrada de este análisis.
    metricas, _, _ = ejecutar_poc('spark', 1, cfg_fixture)
    detalle = Path(metricas['ruta_salida']) / 'detalle'
    spark = crear_sesion(cfg_fixture)
    try:
        planes = planes_ejecucion(spark, detalle, cfg_fixture.ruta_catalogo_tipos)
        assert planes['poda_particiones'] and planes['predicado_empujado'] and planes['broadcast_join']
        formatos = comparar_formatos(spark, detalle, tmp_path / 'formatos', repeticiones=1)
        assert [f['formato'] for f in formatos['formatos']] == ['CSV', 'JSON', 'Parquet']
        assert all(f['filas'] == formatos['registros'] for f in formatos['formatos'])
        df = spark.read.parquet(str(detalle))
        nulos = {n['columna']: n['nulos'] for n in perfil_nulos(df)}
        assert nulos['edad'] >= 1  # el fixture trae edades N/A convertidas a nulo, no imputadas
        assert atipicos_iqr(df, 'edad_anios')['atipicos'] >= 0
    finally:
        detener_sesion(spark)

## 1. Configuración del proyecto

Las rutas y los parámetros salen de `config/config.yaml` (Anexo A).

In [ ]:
import inspect
from IPython.display import Markdown, display
from emergencias123.config import cargar_config

def ver_codigo(*objetos):
    """Muestra en pantalla el código fuente de funciones o módulos que se están explicando."""
    display(Markdown('```python\n' + '\n\n'.join(inspect.getsource(o) for o in objetos) + '\n```'))

cfg = cargar_config()
print('Entorno:', cfg.entorno, '| datos:', cfg.dir_datos, '| resultados:', cfg.dir_resultados)

### Recursos y versiones del runtime

In [ ]:
import json, platform, subprocess, importlib.metadata as md, psutil
from datetime import datetime
entorno = {
    'fecha': datetime.now().isoformat(timespec='seconds'),
    'python': platform.python_version(), 'sistema': platform.platform(),
    'java': subprocess.run(['java', '-version'], capture_output=True, text=True).stderr.splitlines()[0],
    'vcpu': os.cpu_count(), 'ram_gb': round(psutil.virtual_memory().total / 2**30, 2),
    **{p: md.version(p) for p in ('pandas', 'polars', 'pyarrow', 'pyspark', 'psutil', 'numpy')},
}
(cfg.dir_salida('mediciones') / 'entorno_colab.json').write_text(json.dumps(entorno, indent=2, ensure_ascii=False))
entorno

## 2. Datos de entrada

Primero se garantiza que los 3 CSV estén en `BigData_U2/datos/` y sean los esperados; después se perfilan con la
biblioteca estándar (sin usar ninguna de las herramientas comparadas) y se deriva el catálogo de tipos.

In [ ]:
import pandas as pd
from emergencias123.adquisicion import asegurar_datos
origen_datos = asegurar_datos(cfg)
pd.DataFrame([{'archivo': k, 'origen': v['origen'], 'sha256': v['sha256'][:16] + '…'} for k, v in origen_datos.items()])

In [ ]:
from emergencias123.inventario import perfilar, a_markdown
from emergencias123.catalogo import derivar_catalogo, leer_catalogo, leer_valores
inv = perfilar(cfg)
inv['origen_datos'] = origen_datos
(cfg.dir_salida('mediciones') / 'inventario_datos.json').write_text(json.dumps(inv, ensure_ascii=False, indent=2))
derivado = derivar_catalogo(leer_valores(cfg.rutas_csv, 'TIPO_INCIDENTE', cfg.datos['separador'], cfg.datos['codificacion']))
versionado = leer_catalogo(cfg.ruta_catalogo_tipos)
assert [(c['codigo_tipo'], c['descripcion_tipo']) for c in derivado] == \
       [(c['codigo_tipo'], c['descripcion_tipo']) for c in versionado], 'El catálogo no coincide con los datos'
print(f"Catálogo derivado de los datos (no oficial): {len(derivado)} códigos, coincide con la tabla de la parte 1")
_ = (cfg.dir_salida('mediciones') / 'inventario_datos.md').write_text(
    a_markdown(inv, derivado, {'Entorno': 'Google Colab', **{k: entorno[k] for k in ('python', 'pandas', 'polars', 'pyspark', 'vcpu', 'ram_gb')}}),
    encoding='utf-8')
pd.DataFrame([{k: a[k] for k in ('archivo', 'periodo', 'bytes', 'registros', 'fecha_min', 'fecha_max',
                                  'fuera_del_mes_del_archivo', 'duplicados_exactos')} for a in inv['archivos']])

## 3. Actividad 1 — Benchmark Pandas vs Polars vs PySpark

Se comparan las tres herramientas con **los mismos 3 CSV** y **la misma lógica**:

1. **Carga:** leer los 3 CSV y combinarlos (todas las columnas como texto).
2. **Filtrado:** localidades `USAQUEN` y `CHAPINERO` tras normalizar el texto.
3. **Agregación:** conteo de emergencias por `codigo_tipo`.
4. **Join:** LEFT JOIN muchos-a-uno con el catálogo de tipos (derivado, no oficial).
5. **Exportación:** Parquet de los registros filtrados y enriquecidos.

**Medición con `measure_performance()`.**
- Cada repetición corre en un **proceso nuevo**; hay 1 calentamiento descartado y 5 repeticiones medidas por
  herramienta y modo, con el **orden de herramientas rotado** en cada ronda.
- Reloj: `time.perf_counter_ns()` (monotónico; `time.time()` puede saltar con ajustes del reloj del sistema).
- Memoria muestreada cada 20 ms: `psutil.virtual_memory().percent` (% de RAM del sistema, la métrica de la guía)
  y el **RSS del proceso más sus hijos**, que incluye la JVM de Spark.
- **Líneas de código:** líneas no vacías ni de comentario entre los marcadores `# >>> op:` / `# <<< op:`.

**Modos** (Spark y Polars Lazy son perezosos: sin una acción no hay trabajo real).
- `por_operacion`: cada operación se materializa por separado (Spark con `cache()+count()` en cada frontera).
- `extremo_a_extremo`: el pipeline completo con la mínima materialización (Polars `scan_csv` + `collect_all`;
  Spark, un DAG con dos acciones).
- `extremo_a_extremo_caliente` (complementario): segunda pasada en el mismo proceso; separa el coste de una JVM
  recién creada del régimen estable. El arranque de la `SparkSession` se mide aparte como `inicio_sesion`.

### 3.0 Cómo se mide: `measure_performance()` y las 5 operaciones

Estas son las piezas centrales de la comparación. `measure_performance()` ejecuta una operación o un pipeline completo
y devuelve el tiempo (`perf_counter_ns`), el pico de memoria muestreado y las filas de entrada y salida;
`MemorySampler` es el hilo que toma las muestras de memoria cada 20 ms.

In [ ]:
from emergencias123 import benchmark, memoria
ver_codigo(benchmark.measure_performance, memoria.MemorySampler)

**Las cinco operaciones en cada herramienta** (carga, filtrado, agregación, join y exportación). Las líneas entre los
marcadores `# >>> op:` y `# <<< op:` son las que cuenta la complejidad de código de la sección 3.5.

In [ ]:
from emergencias123 import ops_pandas, ops_polars, ops_spark
print('PANDAS')
ver_codigo(ops_pandas.carga, ops_pandas.filtrado, ops_pandas.agregacion, ops_pandas.join, ops_pandas.exportacion)

In [ ]:
print('POLARS')
ver_codigo(ops_polars.carga, ops_polars.filtrado, ops_polars.agregacion, ops_polars.join, ops_polars.exportacion)

In [ ]:
print('PYSPARK')
ver_codigo(ops_spark.carga, ops_spark.filtrado, ops_spark.agregacion, ops_spark.join, ops_spark.exportacion)

### 3.1 Ejecución del benchmark

Para no medir la latencia variable del sistema de archivos de Drive, el benchmark lee una **copia de los CSV en
el disco local del VM** (`/content/u2_bench`) y escribe allí sus Parquet; las tres herramientas usan los mismos
archivos. Solo las mediciones crudas se guardan en Drive. El PoC (parte 4) sí lee directamente desde Drive,
como pide la guía.

In [ ]:
import shutil, time
from pathlib import Path
from emergencias123.benchmark import ejecutar_benchmark, plan_benchmark

LOCAL = Path('/content/u2_bench')
shutil.rmtree(LOCAL, ignore_errors=True)
(LOCAL / 'datos').mkdir(parents=True)
for p in cfg.rutas_csv:
    shutil.copy2(p, LOCAL / 'datos' / p.name)
os.environ['U2_DIR_DATOS'] = str(LOCAL / 'datos')            # lo heredan los subprocesos de medición
os.environ['U2_DIR_RESULTADOS'] = str(LOCAL / 'resultados')
cfg_bench = cargar_config()
salida = LOCAL / 'resultados' / 'mediciones' / 'benchmark.jsonl'
plan = plan_benchmark(cfg_bench)
print(f'{len(plan)} ejecuciones (incluye calentamientos), cada una en un proceso nuevo')

t0 = time.perf_counter()
def progreso(i, n, r):
    marca = 'OK ' if r['codigo_salida'] == 0 else 'ERR'
    print(f"[{i:2d}/{n}] {marca} {r['herramienta']:6s} {r['modo']:26s} e{r['ejecucion']}"
          f"{' (calentamiento)' if r['calentamiento'] else ''} {r['segundos_proceso']:.1f} s", flush=True)
    if r['stderr_final']:
        print(r['stderr_final'])
res = ejecutar_benchmark(cfg_bench, salida, al_avanzar=progreso)
del os.environ['U2_DIR_DATOS'], os.environ['U2_DIR_RESULTADOS']  # el resto del notebook vuelve a Drive
print(f'Duración total: {time.perf_counter() - t0:.0f} s; fallidas: {sum(r["codigo_salida"] != 0 for r in res)}')
shutil.copy2(salida, cfg.dir_salida('mediciones') / 'benchmark.jsonl')
print('Mediciones crudas guardadas en', cfg.dir_salida('mediciones') / 'benchmark.jsonl')

### 3.2 Resultados por operación (modo `por_operacion`, calentamiento excluido)

In [ ]:
from emergencias123.reportes.graficos import cargar_mediciones
crudo = pd.read_json(salida, lines=True)
print('Registros:', len(crudo), '| errores:', int((crudo['estado'] != 'ok').sum()),
      '| calentamientos excluidos:', int(crudo['calentamiento'].sum()))
med = cargar_mediciones(salida)
ops = ['inicio_sesion', 'carga', 'filtrado', 'agregacion', 'join', 'exportacion', 'total', 'total_con_arranque']
t = (med[med['modo'] == 'por_operacion'].groupby(['herramienta', 'operacion'])
     .agg(seg_media=('segundos', 'mean'), seg_desv=('segundos', 'std'), n=('segundos', 'size'),
          rss_pico_mb=('rss_pico_mb', 'mean'), mem_sis_pico_pct=('mem_sis_pico_pct', 'mean'),
          filas_entrada=('filas_entrada', 'first'), filas_salida=('filas_salida', 'first'), loc=('loc', 'first'))
     .reset_index())
t['operacion'] = pd.Categorical(t['operacion'], ops, ordered=True)
t.sort_values(['herramienta', 'operacion']).round(4)

### 3.3 Totales de la carga a la exportación y memoria

In [ ]:
tot = (med[med['operacion'].isin(['total', 'total_con_arranque'])]
       .groupby(['modo', 'herramienta', 'operacion'])
       .agg(seg_media=('segundos', 'mean'), seg_desv=('segundos', 'std'), seg_min=('segundos', 'min'),
            seg_max=('segundos', 'max'), mem_sis_base_pct=('mem_sis_base_pct', 'mean'),
            mem_sis_pico_pct=('mem_sis_pico_pct', 'mean'), rss_pico_mb=('rss_pico_mb', 'mean'), n=('segundos', 'size'))
       .reset_index())
es_total = tot['operacion'] == 'total'
tot.loc[es_total, 'veces_vs_mas_rapida'] = (tot.loc[es_total, 'seg_media']
                                           / tot[es_total].groupby('modo')['seg_media'].transform('min'))
tot.round(3)

In [ ]:
from IPython.display import Image, display
from emergencias123.reportes.graficos import grafico_operaciones, grafico_totales
GRAF = Path('/content/u2_graficos'); GRAF.mkdir(exist_ok=True)   # vista previa; las figuras finales las genera el notebook de informes
for f in (grafico_operaciones(med, GRAF / 'benchmark_operaciones.png'),
          grafico_totales(med, GRAF / 'benchmark_totales_memoria.png')):
    display(Image(filename=str(f), width=1100))

### 3.4 Equivalencia lógica entre herramientas

Se releen los Parquet de la última repetición de cada herramienta y se comparan filas, esquema lógico, conteo por
`codigo_tipo` y la huella SHA-256 del contenido canónico. También se comprueba que el join no multiplique
registros (filas antes = filas después) y que no queden códigos sin coincidencia en el catálogo.

In [ ]:
from emergencias123.equivalencia import comparar
n = cfg_bench.benchmark['repeticiones']
equivalencia = {}
for modo in cfg_bench.benchmark['modos']:
    rutas = {h: cfg_bench.dir_resultados / 'parquet' / 'benchmark' / h / modo / f'e{n}' for h in cfg_bench.benchmark['herramientas']}
    r = comparar(rutas)
    equivalencia[modo] = r
    print(modo, '->', r['criterios'], '| filas:', {h: x['filas'] for h, x in r['resumenes'].items()},
          '| sha256:', {h: x['sha256'][:12] for h, x in r['resumenes'].items()})
join = med[med['operacion'] == 'join']
print('Join sin multiplicar filas:', bool((join['filas_entrada'] == join['filas_salida']).all()),
      '| filas sin coincidencia en el catálogo:', int(join['sin_coincidencia'].max()))
agg = {json.dumps(x, sort_keys=True) for x in med['resultado_agregacion'].dropna()}
print('Agregaciones distintas entre todas las repeticiones y herramientas:', len(agg))
_ = (cfg.dir_salida('mediciones') / 'equivalencia_benchmark.json').write_text(json.dumps(equivalencia, indent=2, ensure_ascii=False))

### 3.5 Complejidad del código (líneas por operación)

In [ ]:
from emergencias123.loc import loc_por_herramienta
loc_ops = loc_por_herramienta()
_ = (cfg.dir_salida('mediciones') / 'loc_benchmark.json').write_text(json.dumps(loc_ops, indent=2))
loc = pd.DataFrame(loc_ops)
loc.loc['TOTAL'] = loc.sum()
loc

### 3.6 Síntesis calculada a partir de las mediciones

In [ ]:
e2e = tot[(tot['modo'] == 'extremo_a_extremo') & (tot['operacion'] == 'total')].set_index('herramienta')
mas_rapida = e2e['seg_media'].idxmin(); menor_mem = e2e['rss_pico_mb'].idxmin()
print(f"Más rápida (extremo a extremo): {mas_rapida} ({e2e.loc[mas_rapida, 'seg_media']:.3f} s)")
for h in e2e.index:
    print(f"  {h:6s}: {e2e.loc[h, 'seg_media']:.3f} s  ({e2e.loc[h, 'seg_media'] / e2e.loc[mas_rapida, 'seg_media']:.1f}x)  "
          f"RSS pico {e2e.loc[h, 'rss_pico_mb']:.0f} MB  memoria sistema pico {e2e.loc[h, 'mem_sis_pico_pct']:.1f} %")
arr = med[(med['herramienta'] == 'spark') & (med['operacion'] == 'inicio_sesion')]['segundos']
print(f"Arranque de SparkSession (aparte): {arr.mean():.2f} s de media")
print(f"Menor memoria (RSS proceso+hijos): {menor_mem}")
cal = tot[(tot['modo'] == 'extremo_a_extremo_caliente') & (tot['operacion'] == 'total')].set_index('herramienta')
print('Proceso frío vs caliente (complementario; misma lógica, 2.ª pasada en el mismo proceso):')
for h in cal.index:
    print(f"  {h:6s}: frío {e2e.loc[h, 'seg_media']:.3f} s -> caliente {cal.loc[h, 'seg_media']:.3f} s "
          f"({e2e.loc[h, 'seg_media'] / cal.loc[h, 'seg_media']:.1f}x)")

### 3.7 Limitaciones de la medición y matriz de decisión

- **Volumen pequeño** (~3 MB, 27 434 filas): dominan los costes fijos (arranque de la JVM, planificación y
  serialización de Spark), y el resultado no se puede extrapolar linealmente a volúmenes grandes.
- **Colab es un único nodo** (2 vCPU, ~12.7 GB): Spark corre en modo `local[2]`, no en un clúster.
- El **% de memoria del sistema** incluye el kernel del notebook y los servicios de Colab; el RSS suma páginas
  compartidas y el muestreo cada 20 ms puede perder picos más breves.
- Tras el calentamiento, la caché de páginas del sistema operativo queda igual de caliente para las tres herramientas.

**Matriz de decisión.** Con estas mediciones, el notebook de informes construye la matriz ponderada (rendimiento
30 %, facilidad de uso 25 %, escalabilidad 20 %, recursos 15 %, curva de aprendizaje 10 %) **con fórmulas de
Excel** en `Actividad_1/reporte_comparativo_benchmark.xlsx`, y la justifica en `analisis_decision_tecnologica.docx`.
PySpark es la herramienta principal por exigencia de la actividad y por su escalabilidad horizontal, aunque con
este volumen no sea la más rápida.

## 4. Actividad 2 — PoC en PySpark

Pipeline funcional en PySpark que procesa los **3 meses** leyendo los CSV **desde Google Drive**. Se ejecuta
**3 veces** para obtener métricas promedio y se compara con una implementación en **Pandas de alcance
equivalente** (mismas reglas, mismas salidas).

**Pasos del pipeline.**
1. Comprobar archivos, encabezados requeridos y filas físicas.
2. Ingesta con **esquema explícito** (todo texto y `_corrupt_record`), en modo PERMISSIVE.
3. Procedencia: `archivo_origen` y `periodo` (mes del archivo, desde la configuración).
4. Nulos: `N/A` y vacío pasan a nulo, **sin imputar**.
5. Tipos: fechas con `try_to_timestamp`, enteros con `try_cast` (Spark 4 tiene ANSI activo), edad en años según UNIDAD.
6. Estandarización: localidad (sin tildes, cruzada con la tabla oficial), código de tipo, prioridad y género.
7. **Observaciones de calidad** en `observacion_calidad` (el registro se conserva): registro corrupto, incidente
   nulo, fecha inválida, localidad sin dato o fuera del catálogo, código de localidad inconsistente, tipo nulo y
   edad fuera de 0–120.
8. **Duplicados exactos:** se eliminan solo las copias con el 100 % de los atributos iguales (se conserva la
   primera aparición) y se escriben en `duplicados_eliminados/`. Los incidentes con varios despachos son
   legítimos y solo se informan.
9. **Broadcast join** con el catálogo de tipos (23 filas, derivado, no oficial).
10. Agregados: periodo×localidad×tipo, periodo×prioridad y conteo diario.
11. **Parquet** particionado por `periodo`, ordenado por localidad dentro de cada partición.
12. Relectura de los Parquet con un lector independiente (pyarrow) y log de validación `.txt`.

**Tasas.**
- **Tasa de éxito de validaciones (%)** = controles aprobados / controles evaluados × 100 (evaluados =
  bloqueantes + advertencias; los informativos no entran).
- **Registros conservados (%)** = conservados / leídos × 100; es otra métrica y se reporta aparte.

### Código del pipeline y de los controles de calidad

`procesar()` implementa los 12 pasos; `evaluar()` calcula los controles V01–V20 a partir de los hechos medidos.

In [ ]:
from emergencias123 import pipeline_spark, validaciones
ver_codigo(pipeline_spark.procesar, validaciones.evaluar)

### 4.1 Configuración de Spark ajustada a lo detectado

In [ ]:
from emergencias123.spark_session import configuracion_spark
print('Modo: local en un único nodo de Colab (no es un clúster multinodo)')
configuracion_spark(cfg)

### 4.2 Ejecución: 3 veces Spark y 3 veces Pandas (mismo alcance)

Las herramientas se alternan en cada ronda. Cada ejecución escribe en `resultados/parquet/poc/<herramienta>/e<N>/`
(Drive), agrega sus métricas a `resultados/mediciones/poc.jsonl` y su log a
`resultados/logs/validacion_poc_<herramienta>.txt`.

In [ ]:
from emergencias123.poc import ejecutar_poc_completo, ejecutar_subproceso
for ruta in [cfg.dir_salida('mediciones') / 'poc.jsonl', *cfg.dir_salida('logs').glob('validacion_poc_*.txt')]:
    ruta.unlink(missing_ok=True)   # cada ejecución completa del notebook reemplaza la anterior
shutil.rmtree(cfg.dir_resultados / 'parquet', ignore_errors=True)
t0 = time.perf_counter()
def progreso(r):
    print(f"{'OK ' if r['codigo_salida'] == 0 else 'ERR'} {r['herramienta']:6s} e{r['ejecucion']} "
          f"{r['segundos_proceso']:.1f} s -> {r['salida'].strip()[-300:]}", flush=True)
    if r['stderr_final']:
        print(r['stderr_final'])
res = ejecutar_poc_completo(cfg, al_avanzar=progreso)
print('Variante de particionamiento periodo/localidad (1 ejecución, para documentar archivos pequeños):')
progreso(ejecutar_subproceso('spark', 1, particionamiento='periodo_localidad', etiqueta='variante_periodo_localidad'))
print(f'Duración total: {time.perf_counter() - t0:.0f} s')

### 4.3 Métricas de las 3 ejecuciones y promedios

In [ ]:
poc = pd.read_json(cfg.dir_resultados / 'mediciones' / 'poc.jsonl', lines=True)
principal = poc[poc['etiqueta'].isna() & (poc['estado'] == 'ok')]
cols = ['herramienta', 'ejecucion', 'tiempo_total_s', 'tiempo_inicio_sesion_s', 'mem_sis_base_pct', 'mem_sis_pico_pct',
        'rss_pico_mb', 'throughput_reg_min', 'tasa_validaciones_pct', 'registros_validos_pct',
        'leidos', 'aceptados', 'rechazados', 'errores']
display(principal[cols].rename(columns={'registros_validos_pct': 'registros_conservados_pct', 'aceptados': 'conservados',
                                        'rechazados': 'duplicados_eliminados'}).sort_values(['herramienta', 'ejecucion']))
prom = principal.groupby('herramienta')[cols[2:]].mean().round(3)
prom

In [ ]:
s, p = prom.loc['spark'], prom.loc['pandas']
print(f"Tiempo total medio:  Spark {s.tiempo_total_s:.2f} s vs Pandas {p.tiempo_total_s:.2f} s "
      f"-> Spark/Pandas = {s.tiempo_total_s / p.tiempo_total_s:.2f}x")
print(f"Memoria sistema pico: Spark {s.mem_sis_pico_pct:.1f} % vs Pandas {p.mem_sis_pico_pct:.1f} %")
print(f"RSS pico proceso+JVM: Spark {s.rss_pico_mb:.0f} MB vs Pandas {p.rss_pico_mb:.0f} MB "
      f"-> {s.rss_pico_mb / p.rss_pico_mb:.1f}x")
print(f"Throughput: Spark {s.throughput_reg_min:,.0f} vs Pandas {p.throughput_reg_min:,.0f} registros/min")
print(f"Arranque de SparkSession (no incluido en el total): {s.tiempo_inicio_sesion_s:.2f} s")
print(f"Registros: {s.leidos:,.0f} leídos, {s.aceptados:,.0f} conservados, {s.rechazados:,.0f} copias duplicadas exactas eliminadas")
print('Fallos de ejecución:', int((poc['estado'] != 'ok').sum()))

### 4.4 Log de validación (Spark)

In [ ]:
print((cfg.dir_resultados / 'logs' / 'validacion_poc_spark.txt').read_text(encoding='utf-8'))

### 4.5 Inspección de los Parquet exportados en Drive (con Spark)

In [ ]:
from emergencias123.spark_session import crear_sesion
from pyspark.sql import functions as F
spark = crear_sesion(cfg)
ult = cfg.dir_resultados / 'parquet' / 'poc' / 'spark' / f"e{cfg.poc['ejecuciones']}"
detalle = spark.read.parquet(str(ult / 'detalle'))
print('Ubicación en Drive:', ult)
detalle.printSchema()
print('Filas:', detalle.count())
detalle.groupBy('periodo').count().orderBy('periodo').show()
detalle.groupBy('observacion_calidad').count().show(truncate=False)
detalle.select('numero_incidente', 'fecha_inicio', 'periodo', 'localidad_norm', 'codigo_tipo', 'descripcion_tipo',
               'prioridad', 'edad_anios', 'observacion_calidad').show(5, truncate=False)
spark.read.parquet(str(ult / 'agregados' / 'periodo_localidad_tipo')).orderBy(F.desc('conteo')).show(8, truncate=False)
print('Copias duplicadas exactas eliminadas (registradas aparte):')
spark.read.parquet(str(ult / 'duplicados_eliminados')).show(5, truncate=False)

## 5. Particionamiento y escalabilidad

### 5.1 `periodo` frente a `periodo/localidad`

Con ~9 100 registros por mes, particionar también por localidad (19–20 valores) produce decenas de archivos de
pocos KB: más metadatos y aperturas de archivo en cada lectura, sin beneficio real de poda.

In [ ]:
var = poc[poc['etiqueta'] == 'variante_periodo_localidad']
comp = pd.concat([principal[principal['herramienta'] == 'spark'].tail(1).assign(estrategia='periodo'),
                  var.assign(estrategia='periodo/localidad')])
comp['kb_por_archivo'] = comp['parquet_bytes'] / comp['parquet_archivos'] / 1024
comp[['estrategia', 'parquet_archivos', 'parquet_bytes', 'kb_por_archivo', 'tiempo_total_s']].round(1)

### 5.2 Volumen replicado (datos SINTÉTICOS)

El PoC (Spark y Pandas, 1 ejecución cada uno) se repite con los 3 CSV **replicados 4 y 8 veces** (~110 mil y
~220 mil registros, equivalentes en filas a ~1 y ~2 años). Cada réplica lleva un sufijo `-R<k>` en
`NUMERO_INCIDENTE` para que no sea un duplicado exacto de la original; si lo fuera, la regla de limpieza la
eliminaría y el volumen procesado no crecería. Los datos están en el disco del VM y las métricas se guardan en
`resultados/mediciones/escalabilidad.jsonl`. Estas cifras miden **volumen**, no la variabilidad real de 1–2 años.

In [ ]:
from emergencias123.poc import ejecutar_escalabilidad
ruta_esc = cfg.dir_salida('mediciones') / 'escalabilidad.jsonl'
ruta_esc.unlink(missing_ok=True)
def progreso_esc(r):
    print(f"{'OK ' if r['codigo_salida'] == 0 else 'ERR'} x{r['factor']} {r['herramienta']:6s} {r['segundos_proceso']:.1f} s",
          flush=True)
    if r['stderr_final']:
        print(r['stderr_final'])
ejecutar_escalabilidad(cfg, al_avanzar=progreso_esc)
esc = pd.read_json(ruta_esc, lines=True)
base = principal.groupby('herramienta')[['leidos', 'aceptados', 'tiempo_total_s', 'rss_pico_mb', 'mem_sis_pico_pct']].mean().reset_index()
base['factor_replica'] = 1
tabla_esc = pd.concat([base, esc[['herramienta', 'factor_replica', 'leidos', 'aceptados', 'tiempo_total_s', 'rss_pico_mb',
                                  'mem_sis_pico_pct']]]).sort_values(['herramienta', 'factor_replica'])
tabla_esc['throughput_reg_min'] = tabla_esc['leidos'] / (tabla_esc['tiempo_total_s'] / 60)
tabla_esc.round(2)

## 6. Optimización y calidad con los patrones del curso

Se trasladan al caso de la Línea 123 cuatro prácticas de los notebooks de ejemplo del curso:
- **Planes de ejecución (`explain`)** sobre el Parquet exportado: *poda de particiones* (`PartitionFilters` sobre
  `periodo`), *predicado empujado* al lector (`PushedFilters` sobre `localidad_norm`) y *broadcast join*.
- **Formatos CSV, JSON y Parquet:** tamaño y tiempo de lectura más agregación (mediana de 3 ejecuciones).
- **Perfil de nulos** del detalle, con una sola agregación.
- **Atípicos por IQR** en `edad_anios`: solo diagnóstico; no se imputa ni se elimina nada.

In [ ]:
from emergencias123.optimizacion import atipicos_iqr, comparar_formatos, perfil_nulos, planes_ejecucion
planes = planes_ejecucion(spark, ult / 'detalle', cfg.ruta_catalogo_tipos)
print('Poda de particiones:', planes['poda_particiones'], '| Predicado empujado:', planes['predicado_empujado'],
      '| Broadcast join:', planes['broadcast_join'])
print(planes['plan_filtro'][:1800])
print(planes['plan_union'][:1200])
formatos = comparar_formatos(spark, ult / 'detalle', '/content/u2_formatos')
display(pd.DataFrame(formatos['formatos']))
detalle_df = spark.read.parquet(str(ult / 'detalle'))
nulos = pd.DataFrame(perfil_nulos(detalle_df)).query('nulos > 0')
display(nulos)
atipicos = atipicos_iqr(detalle_df, 'edad_anios')
print('Atípicos por IQR en edad_anios (informativo):', atipicos)
optim = {'planes': {k: v for k, v in planes.items() if not k.startswith('plan_')}, 'formatos': formatos,
         'nulos': nulos.to_dict('records'), 'atipicos_edad': atipicos}
(cfg.dir_salida('mediciones') / 'optimizacion_poc.json').write_text(json.dumps(optim, indent=2, ensure_ascii=False))
_ = (cfg.dir_salida('mediciones') / 'planes_ejecucion.txt').write_text(planes['plan_filtro'] + '\n\n' + planes['plan_union'])

### 6.1 Gráfico comparativo frente a Pandas

In [ ]:
from emergencias123.reportes.graficos import grafico_poc
ruta = grafico_poc(principal, GRAF / 'comparativo_poc_vs_pandas.png')
spark.stop()
Image(filename=str(ruta), width=1000)

## 7. Pruebas automáticas (pytest)

Las pruebas verifican la normalización y el catálogo, la adquisición con verificación de huella, la equivalencia
de Pandas, Polars y PySpark, las reglas del pipeline (solo se eliminan duplicados exactos; los nulos se conservan
marcados), la fórmula de las tasas, la relectura de los Parquet y la replicación del experimento de escalabilidad.
Usan un conjunto **sintético** pequeño (cada fila ejercita una regla) y, cuando corresponde, los datos reales.

In [ ]:
import subprocess, tempfile
r = subprocess.run([sys.executable, '-m', 'pytest', 'tests', '-q', '-p', 'no:cacheprovider', '--basetemp',
                    tempfile.mkdtemp()], cwd=LIB, capture_output=True, text=True,
                   env={**os.environ, 'PYTHONPATH': f'{LIB}/src'})
print(r.stdout[-2500:] or r.stderr[-2500:])
assert r.returncode == 0, 'Hay pruebas fallidas'

## 8. Conclusiones y próximos pasos

- Todas las cifras de este notebook provienen de su ejecución en Colab (fecha y versiones en la parte 1).
- Con 3 meses de datos, Pandas y Polars son más rápidos y usan menos memoria que Spark: el arranque de la JVM, la
  planificación y la serialización pesan más que el procesamiento. La justificación de PySpark es la escalabilidad
  horizontal, no el rendimiento con este volumen.
- Colab ejecuta Spark en un solo nodo. Un despliegue distribuido (Databricks o un clúster) cambia el balance entre
  los costes fijos y el paralelismo.
- **Siguiente paso (sección 9):** este mismo notebook ejecuta a continuación `U2_informes_y_entregables.ipynb`,
  que genera los informes de las dos actividades a partir de lo que dejó en `BigData_U2/resultados/`, y ordena la
  carpeta `BigData_U2` en Drive.

**Preguntas de reflexión.** ¿Desde qué volumen de datos compensa el coste fijo de Spark? ¿Qué control de calidad
debería detener la publicación de un mes? ¿Cuándo convendría particionar también por localidad?

**Próximos pasos.** Catálogo oficial de códigos; carga incremental mensual; ejecución en Databricks o en un
clúster con al menos un año de datos reales.

## 9. Informes y carpeta de entregables

Con las mediciones ya guardadas, esta sección ejecuta el notebook `U2_informes_y_entregables.ipynb` (debe estar en la
misma carpeta de Drive). Así, **un solo *Ejecutar todas*** deja listo todo dentro de `MyDrive/BigData_U2/`:
`Actividad_1/`, `Actividad_2/`, `Sustentacion/`, `LEEME.md`, `datos/` y `resultados/`. Los informes no vuelven a medir
nada: leen `resultados/mediciones/`.

**Completa estos tres campos** (bloque de título APA de los Word). Si se dejan entre corchetes, los documentos
salen con el marcador `[...]`.

In [ ]:
AUTOR = '[Nombre del estudiante o del equipo]'
INSTITUCION = '[Nombre de la institución]'
DOCENTE = '[Nombre del docente]'

In [ ]:
import nbformat
ruta_informes = f'{BASE}/U2_informes_y_entregables.ipynb'
assert os.path.exists(ruta_informes), 'Falta U2_informes_y_entregables.ipynb en BigData_U2: súbelo junto a este notebook.'
shell = get_ipython()
for celda in nbformat.read(ruta_informes, as_version=4).cells:
    if celda.cell_type == 'code':
        shell.run_cell(celda.source).raise_error()